# Backend Verification for the iHP Open MPW Shuttles

[D. Mitch Bailey](https://www.linkedin.com/in/mitch-bailey-cvc/), [ShuhariSystem](https://www.shuharisystem.com/)

# Overview
This notebook performs backend LVS (with [klayout](https://github.com/KLayout/klayout) or [magic](https://github.com/RTimothyEdwards/magic)/[netgen](https://github.com/RTimothyEdwards/netgen)), soft-connection checks (with magic/netgen) and/or reliability verfication (with [CVC-RV](https://github.com/d-m-bailey/cvc)) with open source EDA software and systems.

# iHP Open-MPW Data
The 17 submissions to iHp open mpw shuttle for April 2025 listed below are located [here](https://github.com/IHP-GmbH/TO_Apr2025).

* 160GHz_LNA
* 40_GHZ_LOW_NOISE_TIA
* 6502-cpu
* 97_GHZ_LINEAR_TIA
* DC_to_130_GHz_TIA
* GPS_LNA
* Greyhound
* Mixer5GHz
* PA_180GHz
* TTIHP0p2
* TTIHP25a
* VCO_130nm_LSI
* active_L_VCOs
* ascon
* bandgap_ref_cmos
* elemrv-n
* i2c-gpio-expander

Create a file with the initial environment variables.


In [ ]:
%%writefile /content/env
export LOCAL_INSTALL=/content/local
export PATH=$PATH:$LOCAL_INSTALL/bin
export PDK_ROOT=/content/pdks
export PDK=ihp-sg13g2
export PDKPATH=$PDK_ROOT/$PDK
# ddb601a4a4473163e1ed6df416b885df18b4ac03 (2025.01.08)
# export PDK_COMMIT=ddb601a4a4473163e1ed6df416b885df18b4ac03
# cb7daaa8901016cf7c5d272dfa322c41f024931f (2025.07.18)
# export PDK_COMMIT=cb7daaa8901016cf7c5d272dfa322c41f024931f
# b4b2bd803b3e4887c31716f3001a1a776b6d56a9 (2026.01.06)
export PDK_COMMIT=b4b2bd803b3e4887c31716f3001a1a776b6d56a9
export MAGIC_VERSION=8.3.620
export NETGEN_VERSION=1.5.314
export KLAYOUT_DISTRO=Ubuntu-22
export KLAYOUT_DEB=klayout_0.30.7-1_amd64.deb
export EXTRA_CHECK_VERSION=ihp-sg13g2
export CVC_VERSION=master
export DATA_ROOT=/content/data
export CHECK_ROOT=/content/check_data
export CHECK_VERSION=main
export LVS_ROOT=/root/extra_be_checks
export MPW=TO_Apr2025
if [[ -f $DATA_ROOT/project_env ]]; then
  cat $DATA_ROOT/project_env
  source $DATA_ROOT/project_env
fi


# Program installation.
Only needs to be executed once.

This step sets up the pdk and installs magic, klayout, netgen and cvc_rv.

Runtime: 3-4 minutes

Note: klayout LVS rule files are modified as follows:
1. `ptap_holes` calculation is modified to ignore ptap in the sealring. If not, the entire chip becomes `ptap_holes`.
1. `rfmim_sub` is redefined as `rfmim_area` sized by 1um. This creates an artificial connection to the substrate under the `rfcap_cmim` device. The substrate is acutally a non-well area with no connection. `rfmim_sub` defined here, creates a region that connects to surrounding psubstrate tap effectively checking that the device is surrounded by a grounded ptap ring.
1. The rule has been modified to combine both layout and source capacitors with the same custom function. The original used default capacitor combination for the source and custom combination for the layout.

Note: The magic tech file `ihp-sg13g2.tech` from `extra_be_checks/tech/ihp-sg13g2` replaces the pdk tech file and contains the following modifications:
1. Because diffusion is silicided, all diffusion, regardless of implant, is connected. The rules are modified to reflect this.
1. The pcell layout for `npn13G2` does not have metal to emitter contact. The pcells are replaced at the fab with proprietary layouts that have the required contacts. To pass LVS, the tech file has been modified to directly connect `metal1` to the `npn13G2` emitter.
1. The `lvnpnarea` calculation was changed so that parallel devices bases are not merged. Merged bases result in only 1 device being extracted.
1. A `pad` layer connection to `metal7` was added.
1. `MET7TXT` labeling of the `pad` layer was added.
1. Individual `res_metal*` devices added to match klayout extraction.
1. For npn, the `le` and `we` parameters values are swapped. magic extracts the smaller value as `le` while the source has `we` as the smaller value.

Note: netgen setup file `ihp-sg13g2_setup.tcl` is modified as follows:
1. Remove parallel combination of `npn` devices. Simple parallel devices are combined during the extraction process with the `merge conservative` directive.
1. Changed device name `cap_rfcmim` to `rfcmim` to match klayout rules. Simulation requires `cap_rfcmim` so this might require a source netlist change or possibly a netgen equivalency.
1. The wfeed parameter in the source netlist is ignored because magic does not extract it.



In [ ]:
%%shell
cd
cat /content/env
source /content/env

lsb_release -a 2>/dev/null

sudo apt update

if ! command -v ciel; then
  echo "==> Installing ciel..."
  pip install ciel
fi
rm -rf $PDK_ROOT
ciel enable --pdk $PDK $PDK_COMMIT
# patch to remove parallel combination of bipolar devices. parallel devices with matching parameters will be combined during magic extraction.
# patch to change cap_rfcmim to rfcmim.
# patch to remove wfeed parameter from source capacitors because magic does not extract it.
sed -i.bak -e '/npn13/,/circuit2.*delete/s/.*parallel/#&/' \
  -e 's/cap_rfcmim/rfcmim/' \
  -e '/rfcmim/,/circuit2.*delete/s/.*circuit2.*delete.*/& wfeed/' $PDK_ROOT/$PDK/libs.tech/netgen/ihp-sg13g2_setup.tcl

# ignore rule that creates separate taps for labeled wells
# ignore gaurdring when computing ptap_holes
sed -i.bak -e '/^well_patt/s/"/"-/' -e '/^sub_patt/s/"/"-/' \
  -e 's/ptap_holes = .*/ptap_holes = ptap.not(edgeseal_drw).holes/' $PDK_ROOT/$PDK/libs.tech/klayout/tech/lvs/rule_decks/general_derivations.lvs

# fix rfmim_sub
sed -i.bak -e 's/rfmim_sub =.*/rfmim_sub = rfmim_area.sized(1.um)/' $PDK_ROOT/$PDK/libs.tech/klayout/tech/lvs/rule_decks/cap_derivations.lvs

# fix cap reduction
mv $PDK_ROOT/$PDK/libs.tech/klayout/tech/lvs/rule_decks/custom_mim_extractor.lvs $PDK_ROOT/$PDK/libs.tech/klayout/tech/lvs/rule_decks/custom_mim_extractor.lvs.org
awk '\
/Adding extra param.*rfcmim/ { print "    self.combiner = MIMCAPNDeviceCombiner.new"; } \
/if name.downcase.include...rfcmim/ { comment += 1; } \
comment == 1 { $1 = "#" $1; } \
comment == 1 && $1 == "#end" { comment += 1; } \
 { print $0; }' \
  $PDK_ROOT/$PDK/libs.tech/klayout/tech/lvs/rule_decks/custom_mim_extractor.lvs.org > \
  $PDK_ROOT/$PDK/libs.tech/klayout/tech/lvs/rule_decks/custom_mim_extractor.lvs

echo "==> Using pdk $PDK commit $PDK_COMMIT (patched)
"

echo "==> Downloading extra_be_checks $EXTRA_CHECK_VERSION"
rm -rf extra_be_checks
git clone https://github.com/d-m-bailey/extra_be_checks.git -b $EXTRA_CHECK_VERSION
cp -f $PDK_ROOT/$PDK/libs.tech/magic/ihp-sg13g2.tech $PDK_ROOT/$PDK/libs.tech/magic/ihp-sg13g2.tech.org
cp -f $PDK_ROOT/$PDK/libs.tech/magic/ihp-sg13g2-cifin.tech $PDK_ROOT/$PDK/libs.tech/magic/ihp-sg13g2-cifin.tech.org
cp -f $PDK_ROOT/$PDK/libs.tech/magic/ihp-sg13g2-extract.tech $PDK_ROOT/$PDK/libs.tech/magic/ihp-sg13g2-extract.tech.org
# cp extra_be_checks/tech/ihp-sg13g2/ihp-sg13g2.tech $PDK_ROOT/$PDK/libs.tech/magic/ihp-sg13g2.tech
# Exchange npn13g2 l/w params
sed -i.bak -e 's/w1=we l1=le/w1=le l1=we/' $PDK_ROOT/$PDK/libs.tech/magic/ihp-sg13g2.tech
# echo "==> Using extra_be_checks commit $(cd extra_be_checks; git rev-parse HEAD)
# "

if ! command -v netgen; then
  echo "==> Downloading and installing netgen $NETGEN_VERSION"
  git clone https://github.com/RTimothyEdwards/netgen.git --depth=1 -b $NETGEN_VERSION
  cd netgen
  ./configure --prefix=$LOCAL_INSTALL
  make
  make install
  cd
fi
echo "==> Using netgen version $(netgen -batch | awk '/Netgen/ {print $2}')
"

if ! command -v magic; then
  echo "==> Downloading and installing magic $MAGIC_VERSION"
  git clone https://github.com/RTimothyEdwards/magic.git --depth=1 -b $MAGIC_VERSION
  cd magic
  ./configure --prefix=$LOCAL_INSTALL
  make
  make install
  cd
fi
echo "==> Using magic version $(magic -dnull -noc --version)
"

if ! command -v klayout; then
  echo "==> Downloading and installing klayout $KLAYOUT_DEB for $KLAYOUT_DISTRO"
  wget -P /tmp https://www.klayout.org/downloads/$KLAYOUT_DISTRO/$KLAYOUT_DEB
  # sudo apt install /root/$KLAYOUT_DEB
  sudo dpkg -i /tmp/$KLAYOUT_DEB
  sudo apt --fix-broken install
  pip install docopt
  pip install klayout
  cd
fi
echo "==> Using $(klayout -v)
"

if ! command -v cvc_rv; then
  echo "==> Downloading and installing cvc_rv $CVC_VERSION"
  sudo apt install autopoint bison flex
  git clone https://github.com/d-m-bailey/cvc --depth=1 -b $CVC_VERSION
  cd cvc
  autoreconf -vif
  ./configure --prefix=$LOCAL_INSTALL --disable-nls
  make
  make install
  cd
fi
echo "==> Using $(cvc_rv -v)
"

echo "#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~"
echo "pdk $PDK commit $PDK_COMMIT (patched)"
echo "extra_be_checks commit $(cd extra_be_checks; git rev-parse HEAD)"
echo "netgen version $(netgen -batch | awk '/Netgen/ {print $2}')"
echo "magic version $(magic -dnull -noc --version)"
echo "$(klayout -v)"
echo "$(cvc_rv -v)"
echo "#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~"

# Clone the design repo and list the gds, spice and verilog files.

Run to refresh the design data.
The file list is also saved in /content/filelist.txt.

Runtime: 1-2 minutes.

In [ ]:
%%shell
cat /content/env
source /content/env
rm -rf $DATA_ROOT $CHECK_ROOT
git clone https://github.com/d-m-bailey/ihp-mpw-be.git --depth=1 -b $CHECK_VERSION $CHECK_ROOT
git clone https://github.com/IHP-GmbH/$MPW.git --depth=1 $DATA_ROOT
cd data
# uncompress gds, spice, cdl, and verilog files
find . \( -name "*.gds.gz" -o -name "*.spice.gz" -o -name "*.cdl.gz" -o -name "*.v.gz" \) -execdir gunzip -v {} \;
find . \( -name "*.gds.zip" -o -name "*.spice.zip" -o -name "*.cdl.zip" -o -name "*.v.zip" \) -execdir 7z x {} \;
# Create a file listing the size, project and relevant file names.
# Only list files that end with .gds (in gds subdirectories), .spice, .cdl, or .v.
tee /content/filelist.txt <<EOF
   Size Project                   File
------- ------------------------- ---------------------------------------------
EOF
ls -s $(find . \( -name "*.gds" -path "*/gds/*" \) -o -name "*.spice" -o -name "*.cdl" -o -name "*.v") |
  sed -e 's,\./,,' -e 's,/, ,' |
  awk '{printf "%7d %-25s %s\n", $1, $2, $3}' |
  tee -a /content/filelist.txt

# ※ Run cells before this cell for program installation and general setup.<br>Then choose one setup from the 17 designs below.<br>Finally, run the checks and examine the results

General results:
1. When the layout has cells that have the same name as primitive devices, netgen segfaults. <br>➡︎ Flatglob all layout cells with primitive device names.

# 1. 160GHz_LNA

Checks will not work with GDS file in repo.
1. Copy text from `LNA` ports layer 134/25 to top layout.
1. Copy corresponding pins shapes on layer 134/2 from `LNA` to top layout (magic only).
1. Copy Passiv shapes layer 9/0 over `RF_INPUT` and `RF_OUTPUT` pins to the top layout and move to dfpad - 41/0 (magic only).

Results:
*   Passes all checks with the above changes

Set the project, top cell and gds file names.

In [ ]:
%%shell
source /content/env
cat > $DATA_ROOT/project_env <<'EOF'
export PROJECT=160GHz_LNA
export WORK_ROOT=$DATA_ROOT/$PROJECT/work/$PROJECT
export LAYOUT_TOP=FMD_QNC_04_160GHz_LNA
# original data
#export LAYOUT_DATA=$DATA_ROOT/$PROJECT/design_data/gds/FMD_QNC_04_160GHz_LNA.gds
# modified data
export LAYOUT_DATA=$WORK_ROOT/$LAYOUT_TOP.gds
export SPICE_FILE=$CHECK_ROOT/$MPW/$PROJECT/xschem/lvs/$PROJECT.spice
# export SPICE_FILE=$DATA_ROOT/$PROJECT/design_data/xschem/simulations/$PROJECT.spice
export SPICE_FILE=$WORK_ROOT/$PROJECT.spice
export CDL_FILE=$WORK_ROOT/$LAYOUT_TOP.cdl
EOF
cat $DATA_ROOT/project_env
source $DATA_ROOT/project_env
rm -f $DATA_ROOT/work
rm -rf $WORK_ROOT
mkdir -p $WORK_ROOT
ln -s $WORK_ROOT $DATA_ROOT/work
ls -l $DATA_ROOT/work

Create the modified magic source netlist.

In [ ]:
%%shell
cat /content/env
source /content/env
cat > $CDL_FILE <<'EOF'
** sch_path: /home/shr25031/ihp-mpw-be/TO_Apr2025/160GHz_LNA/xschem/160GHz_LNA_klayout.sch
.subckt 160GHz_LNA RF_INPUT GND VCC VBB1 RF_OUTPUT VBB2
*.PININFO RF_INPUT:I GND:B VCC:B VBB1:B RF_OUTPUT:O VBB2:B
C1 RF_INPUT net1 GND rfcmim w=10.0e-6 l=15.0e-6 wfeed=6.0e-6
Q1 net2 net1 GND GND npn13G2 a=6.3e-14 p=1.94e-06 le=900e-9 we=70.0n m=4
Rc1 net2 VCC rsil w=7.5e-6 l=5.0e-6 m=1 b=0 $SUB=GND $r=5.86045272969374
Rb1 net1 VBB1 rhigh w=1.9e-6 l=6.0e-6 m=1 b=0 $SUB=GND $r=4471.30730050934
CDecap1 VBB1 GND cap_cmim w=20.0e-6 l=25.0e-6 m=2
CDecap2 VCC GND cap_cmim w=20.0e-6 l=25.0e-6 m=2
C2 net2 GND GND rfcmim w=32.0e-6 l=3.72e-6 wfeed=3.0e-6
C3 GND net3 GND rfcmim w=5.2e-6 l=2.4e-6 wfeed=3.0e-6
Rb2 net3 VBB1 rhigh w=1.9e-6 l=5.0e-6 m=1 b=0 $SUB=GND $r=3740.12450481041
CDecap3 VBB1 GND cap_cmim w=20.0e-6 l=25.0e-6 m=2
Q2 net4 net3 GND GND npn13G2 a=6.3e-14 p=1.94e-06 le=900e-9 we=70.0n m=4
Rc2 net4 VCC rsil w=7.5e-6 l=5.0e-6 m=1 b=0 $SUB=GND $r=5.86045272969374
CDecap4 VCC GND cap_cmim w=20.0e-6 l=25.0e-6 m=2
C4 net4 GND GND rfcmim w=32.0e-6 l=3.72e-6 wfeed=3.0e-6
C5 GND net5 GND rfcmim w=7.6e-6 l=2.5e-6 wfeed=3.0e-6
Rb3 net5 VBB1 rhigh w=1.9e-6 l=6.0e-6 m=1 b=0 $SUB=GND $r=4471.30730050934
CDecap5 VBB1 GND cap_cmim w=20.0e-6 l=25.0e-6 m=2
Rc3 net6 VCC rsil w=7.5e-6 l=5.0e-6 m=1 b=0 $SUB=GND $r=5.86045272969374
CDecap6 VCC GND cap_cmim w=20.0e-6 l=25.0e-6 m=2
Q3 net6 net5 GND GND npn13G2 a=6.3e-14 p=1.94e-06 le=900e-9 we=70.0n m=2
Rb4 net7 VBB2 rhigh w=2.0e-6 l=6.0e-6 m=1 b=0 $SUB=GND $r=4243.26530612245
CDecap7 VBB2 GND cap_cmim w=20.0e-6 l=25.0e-6 m=4
Rc4 net8 VCC rsil w=7.5e-6 l=5.5e-6 m=1 b=0 $SUB=GND $r=6.32649800266312
CDecap8 VCC GND cap_cmim w=20.0e-6 l=25.0e-6 m=2
Q4 net8 net7 GND GND npn13G2 a=6.3e-14 p=1.94e-06 le=900e-9 we=70.0n m=2
C6 net6 GND GND rfcmim w=32.0e-6 l=12.5e-6 wfeed=3.0e-6
C7 GND net7 GND rfcmim w=5.1e-6 l=2.4e-6 wfeed=3.0e-6
C8 net8 GND GND rfcmim w=32.0e-6 l=6.2e-6 wfeed=3.0e-6
C9 GND RF_OUTPUT GND rfcmim w=3.2e-6 l=2.4e-6 wfeed=3.0e-6
.ends
EOF

sed -i -e "s/\.subckt $PROJECT/.subckt $LAYOUT_TOP/I" $CDL_FILE

Create `lvs_config.json` and update the `LVS_SPICE_FILES` and `LVS_VERILOG_FILES` for every design.

Update the other parameters as needed.

In [ ]:
%%shell
cat /content/env
source /content/env
cat > $WORK_ROOT/lvs_config.json <<EOF
{
  "#STD_CELL_LIBRARY": "sky130_fd_sc_hd",
  "#INCLUDE_CONFIGS": [
    "$LVS_ROOT/tech/$PDK/lvs_config.base.json"
  ],
  "TOP_SOURCE": "$PROJECT",
  "TOP_LAYOUT": "$LAYOUT_TOP",
  "EXTRACT_FLATGLOB": [
    "cmim*",
    "npn13G2*",
    "rfcmim*",
    "rhigh*",
    "rsil*",
    "sealring*",
    "via_stack*",
    "*_stage",
    "GND",
    "LNA"
  ],
  "EXTRACT_ABSTRACT": [ "" ],
  "LVS_FLATTEN": [ "" ],
  "LVS_NOFLATTEN": [ "" ],
  "LVS_IGNORE": [ "" ],
  "LVS_SPICE_FILES": [
    "$SPICE_FILE"
  ],
  "#LVS_VERILOG_FILES": [
    "$VERILOG_FILE"
  ],
  "LAYOUT_FILE": "$LAYOUT_DATA"
}
EOF

Create the cvcrc setup file and power file.

In [ ]:
%%shell
cat /content/env
source /content/env
cat > $WORK_ROOT/cvcrc <<EOF
CVC_TOP = $LAYOUT_TOP
CVC_NETLIST = $WORK_ROOT/ext/$LAYOUT_TOP.cdl.gz
CVC_MODEL_FILE = $LVS_ROOT/tech/ihp-sg13g2/cvc.models
CVC_POWER_FILE = $WORK_ROOT/cvc.power.$LAYOUT_TOP
CVC_REPORT_FILE = $WORK_ROOT/cvc.log
EOF

cat > $WORK_ROOT/cvc.power.$PROJECT <<EOF
GND power 0.0
VCC power 3.3
EOF

Create the klayout CDL file from the magic source netlist.

In [ ]:
%%shell
source /content/env
# sed -e '/^X[CDLMQR]/s/.//' \
#     -e "s/\.subckt $PROJECT/.subckt $LAYOUT_TOP/I" $SPICE_FILE > $CDL_FILE
perl -pe '
    s/ GND npn13G2 / npn13G2 /;
    s/ GND rfcmim / rfcmim /;
    if (/^R.* (?:rsil|rhigh|rppd)/) {
        @w = split;
        $w[-1] =~ s/\$r=//;
        $_ = sprintf "%s\n", join " ", @w[0 .. 2], $w[-1], @w[3 .. $#w-1];
    }
' ${CDL_FILE} > ${SPICE_FILE}
sed -i -e "s/\.subckt $LAYOUT_TOP/.subckt $PROJECT/I" $SPICE_FILE

Modify the layout file to pass LVS. (see explanation above.)

In [ ]:
%%shell
source /content/env
$CHECK_ROOT/scripts/apply_json_to_layout.py \
  --input_file=$DATA_ROOT/$PROJECT/design_data/gds/FMD_QNC_04_160GHz_LNA.gds \
  --json_file=$CHECK_ROOT/$MPW/$PROJECT/gds/$PROJECT.changes.json \
  --output_file=$LAYOUT_DATA

# 2. 40_GHZ_LOW_NOISE_TIA

Checks will not work with GDS file in repo.
1. Copy TopMetal2.text from `TIA` to FDM_QNC_00_LN_TIA
2. Modify GND text position outside TopMetal2.pin
3. Modify text `VCC1`->`VCC3`, `VCC3`->`VCC1`
4. Save as `FMD_QNC_00_40_GHz_Low_Noise_TIA.mod.gds`
5. Upload to `/content/data/40_GHZ_LOW_NOISE_TIA/design_data/gds`

Set the project, top cell and gds file names.

In [ ]:
%%shell
source /content/env
cat > $DATA_ROOT/project_env <<'EOF'
export PROJECT=40_GHZ_LOW_NOISE_TIA
export LAYOUT_TOP=FDM_QNC_00_LN_TIA
export LAYOUT_DATA=design_data/gds/FMD_QNC_00_40_GHz_Low_Noise_TIA.mod.gds
export WORK_ROOT=$DATA_ROOT/$PROJECT/work/$PROJECT
export SPICE_FILE=$CHECK_ROOT/$MPW/$PROJECT/xschem/lvs/$PROJECT.spice
export CDL_FILE=$WORK_ROOT/$LAYOUT_TOP.cdl
EOF
cat $DATA_ROOT/project_env
source $DATA_ROOT/project_env
rm -f $DATA_ROOT/work
rm -rf $WORK_ROOT
mkdir -p $WORK_ROOT
ln -s $WORK_ROOT $DATA_ROOT/work
ls -l $DATA_ROOT/work


Create lvs_config.json and update the LVS_SPICE_FILES and LVS_VERILOG_FILES for every design.

Update the other parameters as needed.

In [ ]:
%%shell
cat /content/env
source /content/env
cat > $WORK_ROOT/lvs_config.json <<EOF
{
  "#STD_CELL_LIBRARY": "sky130_fd_sc_hd",
  "#INCLUDE_CONFIGS": [
    "$LVS_ROOT/tech/$PDK/lvs_config.base.json"
  ],
  "TOP_SOURCE": "$PROJECT",
  "TOP_LAYOUT": "$LAYOUT_TOP",
  "EXTRACT_FLATGLOB": [
    "cmim*",
    "npn13G2*",
    "rppd*",
    "rsil*",
    "sealring*",
    "*_STAGE",
    "DC_PAD_DOWN",
    "DC_PAD_UP",
    "GND",
    "GND_PLANE",
    "INPUT_PAD",
    "OUTPUT_PAD",
    "TIA",
    "VIA_STACK"
  ],
  "EXTRACT_ABSTRACT": [ "" ],
  "LVS_FLATTEN": [ "" ],
  "LVS_NOFLATTEN": [ "" ],
  "LVS_IGNORE": [ "" ],
  "LVS_SPICE_FILES": [
    "$SPICE_FILE"
  ],
  "#LVS_VERILOG_FILES": [
    "$VERILOG_FILE"
  ],
  "LAYOUT_FILE": "$LAYOUT_DATA"
}
EOF

Write cvcrc file

In [ ]:
%%shell
cat /content/env
source /content/env
cat > $WORK_ROOT/cvcrc <<EOF
CVC_TOP = $LAYOUT_TOP
CVC_NETLIST = $WORK_ROOT/ext/$LAYOUT_TOP.cdl.gz
CVC_MODEL_FILE = $LVS_ROOT/tech/ihp-sg13g2/cvc.models
CVC_POWER_FILE = $WORK_ROOT/cvc.power.$LAYOUT_TOP
CVC_REPORT_FILE = $WORK_ROOT/cvc.log
EOF

cat > $WORK_ROOT/cvc.power.$PROJECT <<EOF
GND power 0.0
VCC* power 3.3
RFIN min@0.0 max@3.3
EOF

Create the klayout CDL file from the magic source netlist.

In [ ]:
%%shell
source /content/env
sed -e '/^X[CDLMQR]/s/.//' \
    -e "s/\.subckt $PROJECT/.subckt $LAYOUT_TOP/I" $SPICE_FILE > $CDL_FILE


Results for 40_GHZ_LOW_NOISE_TIA
*   layout has cells that have the same name as primitive devices. This causes netgen to segfault. <br>=> Flatglob all layout cells with primitive device names.
* Top layout does not have text. `TIA` level has text, but also has missing connectivity. <br>=> Copy text from `TIA` ports to top level.
*  `VCC3` text does not overlap pin shape.<br>=> move text.
* `VCC1` an `VCC3` connections are reversed.![40GHz_klayout_lvs.png](TO_Apr2025/40_GHZ_LOW_NOISE_TIA/images/40GHz_klayout_lvs.png?raw=1)
* Top level met7 text on pads is not recognized in magic extraction. For magic, change tech file or switch to PADID. However, PADID is not used in klayout.
* Reported 1 CVC error:
```
    ! Short detected: 1257 to 0 Estimated current: 2.42mA
    /RX$ R rppd l=6u w=3u (r=520)
```

# 3. 6502-cpu


Set the project, top cell and gds file names.

In [ ]:
%%shell
source /content/env
cat > $DATA_ROOT/project_env <<'EOF'
export PROJECT=6502-cpu
export LAYOUT_TOP=cpu_top
export LAYOUT_DATA=design_data/gds/FMD_QNC_18_6502-cpu.2.gds
export WORK_ROOT=$DATA_ROOT/$PROJECT/work/$LAYOUT_TOP
export SPICE_FILE=$WORK_ROOT/$LAYOUT_TOP.spice
export CDL_FILE=$WORK_ROOT/$LAYOUT_TOP.cdl
EOF
cat $DATA_ROOT/project_env
source $DATA_ROOT/project_env
rm -f $DATA_ROOT/work
rm -rf $WORK_ROOT
mkdir -p $WORK_ROOT
ln -s $WORK_ROOT $DATA_ROOT/work
ls -l $DATA_ROOT/work

Create the modified magic source netlist.

In [ ]:
%%shell
cat /content/env
source /content/env
cat > $SPICE_FILE <<EOF
.SUBCKT $LAYOUT_TOP
* no source
.ENDS
EOF

Create lvs_config.json and update the LVS_SPICE_FILES and LVS_VERILOG_FILES for every design.

Update the other parameters as needed.

In [ ]:
%%shell
cat /content/env
source /content/env
cat > $WORK_ROOT/lvs_config.json <<EOF
{
  "#STD_CELL_LIBRARY": "sky130_fd_sc_hd",
  "#INCLUDE_CONFIGS": [
    "$LVS_ROOT/tech/$PDK/lvs_config.base.json"
  ],
  "TOP_SOURCE": "$PROJECT",
  "TOP_LAYOUT": "$LAYOUT_TOP",
  "EXTRACT_FLATGLOB": [
    "sg13g2_Clamp*",
    "sg13g2_Corner",
    "sg13g2_DCNDiode",
    "sg13g2_DCPDiode",
    "sg13g2_Filler*",
    "sg13g2_GateDecode",
    "sg13g2_IOPad*",
    "sg13g2_LevelDown",
    "sg13g2_LevelUp",
    "sg13g2_RCClamp*",
    "sg13g2_SecondaryProtection",
    "sg13g2_io_*",
    "sg13g2_*LevelUpInv",
    "TEXT*",
    "VIA*"
  ],
  "EXTRACT_ABSTRACT": [ "" ],
  "LVS_FLATTEN": [ "" ],
  "LVS_NOFLATTEN": [ "" ],
  "LVS_IGNORE": [ "" ],
  "LVS_SPICE_FILES": [
    "$SPICE_FILE"
  ],
  "#LVS_VERILOG_FILES": [
    "$VERILOG_FILE"
  ],
  "LAYOUT_FILE": "$LAYOUT_DATA"
}
EOF

Write cvcrc file

In [ ]:
%%shell
cat /content/env
source /content/env
cat > $WORK_ROOT/cvcrc <<EOF
CVC_TOP = $LAYOUT_TOP
CVC_NETLIST = $WORK_ROOT/ext/$LAYOUT_TOP.cdl.gz
CVC_MODEL_FILE = $LVS_ROOT/tech/ihp-sg13g2/cvc.models
CVC_POWER_FILE = $WORK_ROOT/cvc.power.$LAYOUT_TOP
CVC_REPORT_FILE = $WORK_ROOT/cvc.log
EOF

cat > $WORK_ROOT/cvc.power.$PROJECT <<EOF
*vss* power 0.0
iovdd* power 3.3
vdd power 1.5
clk input min@0.0 max@3.3
reset_n input min@0.0 max@3.3
EOF

Create the klayout CDL file from the magic source netlist.

In [ ]:
%%shell
source /content/env
sed -e '/^X[CDLMQR]/s/.//' \
    -e "s/\.subckt $PROJECT/.subckt $LAYOUT_TOP/I" $SPICE_FILE > $CDL_FILE


Results for 6502-cpu (incomplete)
*   Missing top level power and ground.<br>=> Add top level pins for `iovss`, `iovdd`, `vdd`, and `vss`.
* Missing source netlist.
* `substrate` layer in `io` cells is not extracted as expected in magic.

# 4. 97_GHZ_LINEAR_TIA

Checks will not work with GDS file in repo.
1. Copy text from `TIA` ports to top level layout
2. Modify position of `GND` text outside TopMetal2.pin rects (5 points)
3. Remove `VB1`, `VB2` m2.pin/text (4 points) in STAGE_1, STAGE_4, BIAS_1, BIAS_2 cells
4. Save as `FMD_QNC_01_97_GHZ_LINEAR_TIA.mod.gds` and upload to `/content/data/97_GHZ_LINEAR_TIA/design_data/gds`.

Set the project, top cell and gds file names.

In [ ]:
%%shell
source /content/env
cat > $DATA_ROOT/project_env <<'EOF'
export PROJECT=97_GHZ_LINEAR_TIA
export LAYOUT_TOP=FMD_QNC_01_LIN_TIA
export LAYOUT_DATA=design_data/gds/FMD_QNC_01_97_GHZ_LINEAR_TIA.mod.gds
export WORK_ROOT=$DATA_ROOT/$PROJECT/work/$PROJECT
export SPICE_FILE=$CHECK_ROOT/$MPW/$PROJECT/xschem/lvs/$PROJECT.spice
export CDL_FILE=$WORK_ROOT/$LAYOUT_TOP.cdl
EOF
cat $DATA_ROOT/project_env
source $DATA_ROOT/project_env
rm -f $DATA_ROOT/work
rm -rf $WORK_ROOT
mkdir -p $WORK_ROOT
ln -s $WORK_ROOT $DATA_ROOT/work
ls -l $DATA_ROOT/work

Create lvs_config.json and update the LVS_SPICE_FILES and LVS_VERILOG_FILES for every design.

Update the other parameters as needed.

In [ ]:
%%shell
cat /content/env
source /content/env
cat > $WORK_ROOT/lvs_config.json <<EOF
{
  "#STD_CELL_LIBRARY": "sky130_fd_sc_hd",
  "#INCLUDE_CONFIGS": [
    "$LVS_ROOT/tech/$PDK/lvs_config.base.json"
  ],
  "TOP_SOURCE": "$PROJECT",
  "TOP_LAYOUT": "$LAYOUT_TOP",
  "EXTRACT_FLATGLOB": [
    "cmim*",
    "npn13G2*",
    "rhigh*",
    "rppd*",
    "rsil*",
    "sealring",
    "BIAS*",
    "DC_PAD_*",
    "FEEDBACK",
    "GND",
    "INPUT_*",
    "NSL",
    "OUTPUT_*",
    "STAGE*",
    "TIA",
    "VIA*"
  ],
  "EXTRACT_ABSTRACT": [ "" ],
  "LVS_FLATTEN": [ "" ],
  "LVS_NOFLATTEN": [ "" ],
  "LVS_IGNORE": [ "" ],
  "LVS_SPICE_FILES": [
    "$SPICE_FILE"
  ],
  "#LVS_VERILOG_FILES": [
    "$VERILOG_FILE"
  ],
  "LAYOUT_FILE": "$LAYOUT_DATA"
}
EOF

Write cvcrc file

In [ ]:
%%shell
cat /content/env
source /content/env
cat > $WORK_ROOT/cvcrc <<EOF
CVC_TOP = $LAYOUT_TOP
CVC_NETLIST = $WORK_ROOT/ext/$LAYOUT_TOP.cdl.gz
CVC_MODEL_FILE = $LVS_ROOT/tech/ihp-sg13g2/cvc.models
CVC_POWER_FILE = $WORK_ROOT/cvc.power.$LAYOUT_TOP
CVC_REPORT_FILE = $WORK_ROOT/cvc.log
EOF

cat > $WORK_ROOT/cvc.power.$PROJECT <<EOF
GND power 0.0
VCC* power 3.3
RFIN min@0.0 max@3.3
EOF


Create the klayout CDL file from the magic source netlist.

In [ ]:
%%shell
source /content/env
sed -e '/^X[CDLMQR]/s/.//' \
    -e "s/\.subckt $PROJECT/.subckt $LAYOUT_TOP/I" $SPICE_FILE > $CDL_FILE


Results for 97_GHZ_LINEAR_TIA
1. pcells with names matching primitive devices should be flattened in magic extraction.
2. Reported 1 CVC error which can be ignored
    ! Checking forward bias diode errors:
    /QX5 Q npn13g2 we=70n le=0.9u M=10 (r=1)
    B: RFIN

# 5. active_L_VCOs

Not ready - do not run.


Set the project, top cell and gds file names.

In [ ]:
%%shell
source /content/env
cat > $DATA_ROOT/project_env <<'EOF'
export PROJECT=active_L_VCOs
export LAYOUT_TOP=
export LAYOUT_DATA=design_data/gds/
export WORK_ROOT=$DATA_ROOT/$PROJECT/work/$LAYOUT_TOP
export SPICE_FILE=$WORK_ROOT/$LAYOUT_TOP.spice
export CDL_FILE=$WORK_ROOT/$LAYOUT_TOP.cdl
EOF
cat $DATA_ROOT/project_env
source $DATA_ROOT/project_env
rm -f $DATA_ROOT/work
rm -rf $WORK_ROOT
mkdir -p $WORK_ROOT
ln -s $WORK_ROOT $DATA_ROOT/work
ls -l $DATA_ROOT/work

Create the modified magic source netlist.

In [ ]:
%%shell
cat /content/env
source /content/env
cat > $SPICE_FILE <<EOF
.SUBCKT $LAYOUT_TOP
* no source
.ENDS
EOF

Create lvs_config.json and update the LVS_SPICE_FILES and LVS_VERILOG_FILES for every design.

Update the other parameters as needed.

In [ ]:
%%shell
cat /content/env
source /content/env
cat > $WORK_ROOT/lvs_config.json <<EOF
{
  "#STD_CELL_LIBRARY": "sky130_fd_sc_hd",
  "#INCLUDE_CONFIGS": [
    "$LVS_ROOT/tech/$PDK/lvs_config.base.json"
  ],
  "TOP_SOURCE": "$PROJECT",
  "TOP_LAYOUT": "$LAYOUT_TOP",
  "EXTRACT_FLATGLOB": [
    "sg13g2_Clamp*",
    "sg13g2_Corner",
    "sg13g2_DCNDiode",
    "sg13g2_DCPDiode",
    "sg13g2_Filler*",
    "sg13g2_GateDecode",
    "sg13g2_IOPad*",
    "sg13g2_LevelDown",
    "sg13g2_LevelUp",
    "sg13g2_RCClamp*",
    "sg13g2_SecondaryProtection",
    "sg13g2_io_*",
    "sg13g2_*LevelUpInv",
    "TEXT*",
    "VIA*"
  ],
  "EXTRACT_ABSTRACT": [ "" ],
  "LVS_FLATTEN": [ "" ],
  "LVS_NOFLATTEN": [ "" ],
  "LVS_IGNORE": [ "" ],
  "LVS_SPICE_FILES": [
    "$SPICE_FILE"
  ],
  "#LVS_VERILOG_FILES": [
    "$VERILOG_FILE"
  ],
  "LAYOUT_FILE": "$LAYOUT_DATA"
}
EOF

Write cvcrc file

In [ ]:
%%shell
cat /content/env
source /content/env
cat > $WORK_ROOT/cvcrc <<EOF
CVC_TOP = $LAYOUT_TOP
CVC_NETLIST = $WORK_ROOT/ext/$LAYOUT_TOP.cdl.gz
CVC_MODEL_FILE = $LVS_ROOT/tech/ihp-sg13g2/cvc.models
CVC_POWER_FILE = $WORK_ROOT/cvc.power.$LAYOUT_TOP
CVC_REPORT_FILE = $WORK_ROOT/cvc.log
EOF

cat > $WORK_ROOT/cvc.power.$PROJECT <<EOF
*vss* power 0.0
iovdd* power 3.3
vdd power 1.5
clk input min@0.0 max@3.3
reset_n input min@0.0 max@3.3
EOF

Create the klayout CDL file from the magic source netlist.

In [ ]:
%%shell
source /content/env
sed -e '/^X[CDLMQR]/s/.//' \
    -e "s/\.subckt $PROJECT/.subckt $LAYOUT_TOP/I" $SPICE_FILE > $CDL_FILE


Results for active_L_VCOs


# 6. ascon

Not ready - do not run.


Set the project, top cell and gds file names.

In [ ]:
%%shell
source /content/env
cat > $DATA_ROOT/project_env <<'EOF'
export PROJECT=ascon
export LAYOUT_TOP=
export LAYOUT_DATA=design_data/gds/
export WORK_ROOT=$DATA_ROOT/$PROJECT/work/$LAYOUT_TOP
export SPICE_FILE=$WORK_ROOT/$LAYOUT_TOP.spice
export CDL_FILE=$WORK_ROOT/$LAYOUT_TOP.cdl
EOF
cat $DATA_ROOT/project_env
source $DATA_ROOT/project_env
rm -f $DATA_ROOT/work
rm -rf $WORK_ROOT
mkdir -p $WORK_ROOT
ln -s $WORK_ROOT $DATA_ROOT/work
ls -l $DATA_ROOT/work

Create the modified magic source netlist.

In [ ]:
%%shell
cat /content/env
source /content/env
cat > $SPICE_FILE <<EOF
.SUBCKT $LAYOUT_TOP
* no source
.ENDS
EOF

Create lvs_config.json and update the LVS_SPICE_FILES and LVS_VERILOG_FILES for every design.

Update the other parameters as needed.

In [ ]:
%%shell
cat /content/env
source /content/env
cat > $WORK_ROOT/lvs_config.json <<EOF
{
  "#STD_CELL_LIBRARY": "sky130_fd_sc_hd",
  "#INCLUDE_CONFIGS": [
    "$LVS_ROOT/tech/$PDK/lvs_config.base.json"
  ],
  "TOP_SOURCE": "$PROJECT",
  "TOP_LAYOUT": "$LAYOUT_TOP",
  "EXTRACT_FLATGLOB": [
    "sg13g2_Clamp*",
    "sg13g2_Corner",
    "sg13g2_DCNDiode",
    "sg13g2_DCPDiode",
    "sg13g2_Filler*",
    "sg13g2_GateDecode",
    "sg13g2_IOPad*",
    "sg13g2_LevelDown",
    "sg13g2_LevelUp",
    "sg13g2_RCClamp*",
    "sg13g2_SecondaryProtection",
    "sg13g2_io_*",
    "sg13g2_*LevelUpInv",
    "TEXT*",
    "VIA*"
  ],
  "EXTRACT_ABSTRACT": [ "" ],
  "LVS_FLATTEN": [ "" ],
  "LVS_NOFLATTEN": [ "" ],
  "LVS_IGNORE": [ "" ],
  "LVS_SPICE_FILES": [
    "$SPICE_FILE"
  ],
  "#LVS_VERILOG_FILES": [
    "$VERILOG_FILE"
  ],
  "LAYOUT_FILE": "$LAYOUT_DATA"
}
EOF

Write cvcrc file

In [ ]:
%%shell
cat /content/env
source /content/env
cat > $WORK_ROOT/cvcrc <<EOF
CVC_TOP = $LAYOUT_TOP
CVC_NETLIST = $WORK_ROOT/ext/$LAYOUT_TOP.cdl.gz
CVC_MODEL_FILE = $LVS_ROOT/tech/ihp-sg13g2/cvc.models
CVC_POWER_FILE = $WORK_ROOT/cvc.power.$LAYOUT_TOP
CVC_REPORT_FILE = $WORK_ROOT/cvc.log
EOF

cat > $WORK_ROOT/cvc.power.$PROJECT <<EOF
*vss* power 0.0
iovdd* power 3.3
vdd power 1.5
clk input min@0.0 max@3.3
reset_n input min@0.0 max@3.3
EOF

Create the klayout CDL file from the magic source netlist.

In [ ]:
%%shell
source /content/env
sed -e '/^X[CDLMQR]/s/.//' \
    -e "s/\.subckt $PROJECT/.subckt $LAYOUT_TOP/I" $SPICE_FILE > $CDL_FILE


Results for ascon


# 7. bandgap_ref_cmos

## Notice:
Please update pdks/ihp-sg13g2/libs.tech/klayout/tech/lvs/rule_decks/custom_combiner.lvs to fix series resistor problem

Set the project, top cell and gds file names.

In [ ]:
%%shell
source /content/env
cat > $DATA_ROOT/project_env <<'EOF'
export PROJECT=bandgap_ref_cmos
export LAYOUT_TOP=full_bandgap
export LAYOUT_DATA=$DATA_ROOT/$PROJECT/design_data/gds/FMD_QNC_15_WeakInvBGR.mod.gds
export WORK_ROOT=$DATA_ROOT/$PROJECT/work/$PROJECT
export SPICE_FILE=$WORK_ROOT/$PROJECT.spice
export CDL_FILE=$WORK_ROOT/$LAYOUT_TOP.cdl
EOF
# cat $DATA_ROOT/project_env
source $DATA_ROOT/project_env
if [[ -f $LAYOUT_DATA ]]; then
  ls -l $LAYOUT_DATA
else
  echo "LAYOUT_DATA=$LAYOUT_DATA does not exist"
  exit 1
fi
rm -f $DATA_ROOT/work
rm -rf $WORK_ROOT
mkdir -p $WORK_ROOT
ln -s $WORK_ROOT $DATA_ROOT/work
ls -l $DATA_ROOT/work

In [ ]:
%%shell
cat /content/env
source /content/env
cat > $SPICE_FILE <<EOF
** sch_path: /Users/home/IHP/TO_Apr2025/bandgap_ref_cmos/design_data/xschem/part_2_full_bgr/bandgap_reference.sch
.subckt bandgap_ref_cmos vdd VBG vss iout
*.PININFO vplus:B v-:B Vo1:B VBG:O
XM8 net1 v- vss vss sg13_lv_nmos l=10u w=150n ng=1 m=1
XM6 net1 net1 vdd vdd sg13_lv_pmos l=1u w=1u ng=1 m=1
XM7 net2 net1 vdd vdd sg13_lv_pmos l=1u w=1u ng=1 m=1
XM9 v- net2 vdd vdd sg13_lv_pmos l=4u w=200n ng=1 m=1
XM1 v- vplus vss vss sg13_lv_nmos l=5u w=7.14u ng=4 m=1
XM2 net3 net3 vss vss sg13_lv_nmos l=5u w=21u ng=8 m=1
XM3 v- Vo1 vdd vdd sg13_lv_pmos l=5u w=15u ng=8 m=1
XM4 vplus Vo1 vdd vdd sg13_lv_pmos l=5u w=15u ng=8 m=1
XM5 VBG Vo1 vdd vdd sg13_lv_pmos l=5u w=16u ng=8 m=1
*XC3 VBG vss cap_cmim w=72.965e-6 l=72.965e-6 m=1
XC3 vdd vss cap_cmim w=5e-6 l=5e-6 m=15
* XR3 net3 vplus rppd w=0.5e-6 l=193.325e-6 m=1 b=0
XR3 net3 vplus rppd w=0.5e-6 l=193.25e-6 m=1 b=0
*XR1 vss vplus rppd w=0.6e-6 l=194.345e-6 m=1 b=0
XR1_1 vss mid rppd w=3.0e-6 l=38.65e-6 m=1 b=0
XR1_2 mid vplus rppd w=0.5e-6 l=154.6e-6 m=1 b=0
*XR2 vss VBG rppd w=0.5e-6 l=192.395e-6 m=1 b=0
XR2 vss VBG rppd w=0.71e-6 l=270.55e-6 m=1 b=0
XC1 net2 vss cap_cmim w=18.2e-6 l=18.2e-6 m=1
x1 vdd iout vplus v- Vo1 vss two_stage_OTA
.ends

* expanding   symbol:  part_1_OTA/two_stage_OTA.sym # of pins=6
** sym_path: /Users/home/IHP/TO_Apr2025/bandgap_ref_cmos/design_data/xschem/part_1_OTA/two_stage_OTA.sym
** sch_path: /Users/home/IHP/TO_Apr2025/bandgap_ref_cmos/design_data/xschem/part_1_OTA/two_stage_OTA.sch
.subckt two_stage_OTA vdd iout vplus v- vout vss
*.PININFO v-:B vplus:B vss:B vdd:B iout:B vout:B
XM4 net3 net1 vss vss sg13_lv_nmos l=9.75u w=720n ng=1 m=1
XM3 net1 net1 vss vss sg13_lv_nmos l=9.75u w=720n ng=1 m=1
XM1 net1 v- net2 vdd sg13_lv_pmos l=3.7u w=3.64u ng=1 m=2
XM_dummy net1 vdd vdd vdd sg13_lv_pmos l=3.7u w=3.64u ng=1 m=2
XM2 net3 vplus net2 vdd sg13_lv_pmos l=3.7u w=3.64u ng=1 m=2
XM_dummy2 vdd vdd net2 vdd sg13_lv_pmos l=3.7u w=3.64u ng=1 m=4
XM_dummy3 net3 vdd vdd vdd sg13_lv_pmos l=3.7u w=3.64u ng=1 m=2
XM5 net2 iout vdd vdd sg13_lv_pmos l=1.95u w=5.3u ng=1 m=1
XM7 vout iout vdd vdd sg13_lv_pmos l=2.08u w=75u ng=8 m=1
XM6 vout net3 vss vss sg13_lv_nmos l=9.75u w=28.8u ng=4 m=1
XM9 iout iout vdd vdd sg13_lv_pmos l=2.08u w=75u ng=8 m=1
XC2 net3 vout cap_cmim w=22.29e-6 l=22.29e-6 m=1
.ends
EOF

Create lvs_config.json and update the LVS_SPICE_FILES and LVS_VERILOG_FILES for every design.

Update the other parameters as needed.

In [ ]:
%%shell
cat /content/env
source /content/env
cat > $WORK_ROOT/lvs_config.json <<EOF
{
  "#STD_CELL_LIBRARY": "sky130_fd_sc_hd",
  "#INCLUDE_CONFIGS": [
    "$LVS_ROOT/tech/$PDK/lvs_config.base.json"
  ],
  "TOP_SOURCE": "$PROJECT",
  "TOP_LAYOUT": "$LAYOUT_TOP",
  "EXTRACT_FLATGLOB": [
    "*_FILL_*"
  ],
  "EXTRACT_ABSTRACT": [ "" ],
  "LVS_FLATTEN": [ "" ],
  "LVS_NOFLATTEN": [ "" ],
  "LVS_IGNORE": [ "" ],
  "LVS_SPICE_FILES": [
    "$SPICE_FILE"
  ],
  "#LVS_VERILOG_FILES": [
    "$VERILOG_FILE"
  ],
  "LAYOUT_FILE": "$LAYOUT_DATA"
}
EOF

Write cvcrc file

In [ ]:
%%shell
cat /content/env
source /content/env
cat > $WORK_ROOT/cvcrc <<EOF
CVC_TOP = $LAYOUT_TOP
CVC_NETLIST = $WORK_ROOT/ext/$LAYOUT_TOP.cdl.gz
CVC_MODEL_FILE = $LVS_ROOT/tech/ihp-sg13g2/cvc.models
CVC_POWER_FILE = $WORK_ROOT/cvc.power.$LAYOUT_TOP
CVC_REPORT_FILE = $WORK_ROOT/cvc.log
EOF

cat > $WORK_ROOT/cvc.power.$PROJECT <<EOF
vss power 0.0
vdd power 1.5
EOF

Create the klayout CDL file from the magic source netlist.

In [ ]:
%%shell
source /content/env
sed -e '/^X[CDLMQR]/s/.//' \
    -e "s/\.subckt $PROJECT/.subckt $LAYOUT_TOP/I" $SPICE_FILE > $CDL_FILE


Results for bandgap_ref_cmos
*   Pad text is on metal7 layer. This is not recognized on pads in magic. Either move the text to a region outside the pad, change the text layer to the PADID layer (41/0), or modify the magic extraction rules to recognize metal7 text on pads.
*   Klayout extraction rules generate n/ptap devices for layout taps annotated with "well"/"sub!" text. The schematic does not include these devices. Either add the devices to the schematic or remove the "well" and "sub!" text from the layout.
*   Ports do not match. Layout ports are vss, vdd, VBG, and iout. Netlist ports are GND, vdd, v+, v-, Vo1, and VBG. iout in the schematic is currently shorted to ground through an ammeter. Suggest removing v+, v-, Vo1 ports in schematic while adding iout port. Suggest changing GND from a global net to a port named vss in the schematic to match the layout.
* Layout has 3 dummy pmos (gate and source connected to vdd) not found in the schematic. Suggest adding them to the schematic. Suggest adding the following to `two_stage_OTA`
```
XM_dummy net1 vdd vdd vdd sg13_lv_pmos l=3.7u w=3.64u ng=1 m=2
XM_dummy2 vdd vdd net2 vdd sg13_lv_pmos l=3.7u w=3.64u ng=1 m=4
XM_dummy3 net3 vdd vdd vdd sg13_lv_pmos l=3.7u w=3.64u ng=1 m=2
```
* cap_cmim connected between VBG and vss in the schematic is connected between vdd and vss in the layout. The schematic size is 72.965 x 72.965 but the layout size is 5x5 m=15.
* R2 schematic size is 0.5/192.395 but layout is 0.71/38.645x7
* R1 schematic size is 0.6/194.345 but layout consists of resistors of different widths. 3.0/38.645 and 0.5/38.645x4. Currently netgen and klayout only reduce resistors with the same widths.

# 8. DC_to_130_GHz_TIA

Not ready - do not run.


Set the project, top cell and gds file names.

In [ ]:
%%shell
source /content/env
cat > $DATA_ROOT/project_env <<'EOF'
export PROJECT=ascon
export LAYOUT_TOP=
export LAYOUT_DATA=design_data/gds/
export WORK_ROOT=$DATA_ROOT/$PROJECT/work/$LAYOUT_TOP
export SPICE_FILE=$WORK_ROOT/$LAYOUT_TOP.spice
export CDL_FILE=$WORK_ROOT/$LAYOUT_TOP.cdl
EOF
cat $DATA_ROOT/project_env
source $DATA_ROOT/project_env
rm -f $DATA_ROOT/work
rm -rf $WORK_ROOT
mkdir -p $WORK_ROOT
ln -s $WORK_ROOT $DATA_ROOT/work
ls -l $DATA_ROOT/work

Create the modified magic source netlist.

In [ ]:
%%shell
cat /content/env
source /content/env
cat > $SPICE_FILE <<EOF
.SUBCKT $LAYOUT_TOP
* no source
.ENDS
EOF

Create lvs_config.json and update the LVS_SPICE_FILES and LVS_VERILOG_FILES for every design.

Update the other parameters as needed.

In [ ]:
%%shell
cat /content/env
source /content/env
cat > $WORK_ROOT/lvs_config.json <<EOF
{
  "#STD_CELL_LIBRARY": "sky130_fd_sc_hd",
  "#INCLUDE_CONFIGS": [
    "$LVS_ROOT/tech/$PDK/lvs_config.base.json"
  ],
  "TOP_SOURCE": "$PROJECT",
  "TOP_LAYOUT": "$LAYOUT_TOP",
  "EXTRACT_FLATGLOB": [
    "sg13g2_Clamp*",
    "sg13g2_Corner",
    "sg13g2_DCNDiode",
    "sg13g2_DCPDiode",
    "sg13g2_Filler*",
    "sg13g2_GateDecode",
    "sg13g2_IOPad*",
    "sg13g2_LevelDown",
    "sg13g2_LevelUp",
    "sg13g2_RCClamp*",
    "sg13g2_SecondaryProtection",
    "sg13g2_io_*",
    "sg13g2_*LevelUpInv",
    "TEXT*",
    "VIA*"
  ],
  "EXTRACT_ABSTRACT": [ "" ],
  "LVS_FLATTEN": [ "" ],
  "LVS_NOFLATTEN": [ "" ],
  "LVS_IGNORE": [ "" ],
  "LVS_SPICE_FILES": [
    "$SPICE_FILE"
  ],
  "#LVS_VERILOG_FILES": [
    "$VERILOG_FILE"
  ],
  "LAYOUT_FILE": "$LAYOUT_DATA"
}
EOF

Write cvcrc file

In [ ]:
%%shell
cat /content/env
source /content/env
cat > $WORK_ROOT/cvcrc <<EOF
CVC_TOP = $LAYOUT_TOP
CVC_NETLIST = $WORK_ROOT/ext/$LAYOUT_TOP.cdl.gz
CVC_MODEL_FILE = $LVS_ROOT/tech/ihp-sg13g2/cvc.models
CVC_POWER_FILE = $WORK_ROOT/cvc.power.$LAYOUT_TOP
CVC_REPORT_FILE = $WORK_ROOT/cvc.log
EOF

cat > $WORK_ROOT/cvc.power.$PROJECT <<EOF
*vss* power 0.0
iovdd* power 3.3
vdd power 1.5
clk input min@0.0 max@3.3
reset_n input min@0.0 max@3.3
EOF

Create the klayout CDL file from the magic source netlist.

In [ ]:
%%shell
source /content/env
sed -e '/^X[CDLMQR]/s/.//' \
    -e "s/\.subckt $PROJECT/.subckt $LAYOUT_TOP/I" $SPICE_FILE > $CDL_FILE


Results for DC_to_130_GHz_TIA


# 9. elemrv-n

Not ready - do not run.


Set the project, top cell and gds file names.

In [ ]:
%%shell
source /content/env
cat > $DATA_ROOT/project_env <<'EOF'
export PROJECT=ascon
export LAYOUT_TOP=
export LAYOUT_DATA=design_data/gds/
export WORK_ROOT=$DATA_ROOT/$PROJECT/work/$LAYOUT_TOP
export SPICE_FILE=$WORK_ROOT/$LAYOUT_TOP.spice
export CDL_FILE=$WORK_ROOT/$LAYOUT_TOP.cdl
EOF
cat $DATA_ROOT/project_env
source $DATA_ROOT/project_env
rm -f $DATA_ROOT/work
rm -rf $WORK_ROOT
mkdir -p $WORK_ROOT
ln -s $WORK_ROOT $DATA_ROOT/work
ls -l $DATA_ROOT/work

Create the modified magic source netlist.

In [ ]:
%%shell
cat /content/env
source /content/env
cat > $SPICE_FILE <<EOF
.SUBCKT $LAYOUT_TOP
* no source
.ENDS
EOF

Create lvs_config.json and update the LVS_SPICE_FILES and LVS_VERILOG_FILES for every design.

Update the other parameters as needed.

In [ ]:
%%shell
cat /content/env
source /content/env
cat > $WORK_ROOT/lvs_config.json <<EOF
{
  "#STD_CELL_LIBRARY": "sky130_fd_sc_hd",
  "#INCLUDE_CONFIGS": [
    "$LVS_ROOT/tech/$PDK/lvs_config.base.json"
  ],
  "TOP_SOURCE": "$PROJECT",
  "TOP_LAYOUT": "$LAYOUT_TOP",
  "EXTRACT_FLATGLOB": [
    "sg13g2_Clamp*",
    "sg13g2_Corner",
    "sg13g2_DCNDiode",
    "sg13g2_DCPDiode",
    "sg13g2_Filler*",
    "sg13g2_GateDecode",
    "sg13g2_IOPad*",
    "sg13g2_LevelDown",
    "sg13g2_LevelUp",
    "sg13g2_RCClamp*",
    "sg13g2_SecondaryProtection",
    "sg13g2_io_*",
    "sg13g2_*LevelUpInv",
    "TEXT*",
    "VIA*"
  ],
  "EXTRACT_ABSTRACT": [ "" ],
  "LVS_FLATTEN": [ "" ],
  "LVS_NOFLATTEN": [ "" ],
  "LVS_IGNORE": [ "" ],
  "LVS_SPICE_FILES": [
    "$SPICE_FILE"
  ],
  "#LVS_VERILOG_FILES": [
    "$VERILOG_FILE"
  ],
  "LAYOUT_FILE": "$LAYOUT_DATA"
}
EOF

Write cvcrc file

In [ ]:
%%shell
cat /content/env
source /content/env
cat > $WORK_ROOT/cvcrc <<EOF
CVC_TOP = $LAYOUT_TOP
CVC_NETLIST = $WORK_ROOT/ext/$LAYOUT_TOP.cdl.gz
CVC_MODEL_FILE = $LVS_ROOT/tech/ihp-sg13g2/cvc.models
CVC_POWER_FILE = $WORK_ROOT/cvc.power.$LAYOUT_TOP
CVC_REPORT_FILE = $WORK_ROOT/cvc.log
EOF

cat > $WORK_ROOT/cvc.power.$PROJECT <<EOF
*vss* power 0.0
iovdd* power 3.3
vdd power 1.5
clk input min@0.0 max@3.3
reset_n input min@0.0 max@3.3
EOF

Create the klayout CDL file from the magic source netlist.

In [ ]:
%%shell
source /content/env
sed -e '/^X[CDLMQR]/s/.//' \
    -e "s/\.subckt $PROJECT/.subckt $LAYOUT_TOP/I" $SPICE_FILE > $CDL_FILE


Results for elemrv-n


# 10. GPS_LNA

Not ready - do not run.


Set the project, top cell and gds file names.

In [ ]:
%%shell
source /content/env
cat > $DATA_ROOT/project_env <<'EOF'
export PROJECT=ascon
export LAYOUT_TOP=
export LAYOUT_DATA=design_data/gds/
export WORK_ROOT=$DATA_ROOT/$PROJECT/work/$LAYOUT_TOP
export SPICE_FILE=$WORK_ROOT/$LAYOUT_TOP.spice
export CDL_FILE=$WORK_ROOT/$LAYOUT_TOP.cdl
EOF
cat $DATA_ROOT/project_env
source $DATA_ROOT/project_env
rm -f $DATA_ROOT/work
rm -rf $WORK_ROOT
mkdir -p $WORK_ROOT
ln -s $WORK_ROOT $DATA_ROOT/work
ls -l $DATA_ROOT/work

Create the modified magic source netlist.

In [ ]:
%%shell
cat /content/env
source /content/env
cat > $SPICE_FILE <<EOF
.SUBCKT $LAYOUT_TOP
* no source
.ENDS
EOF

Create lvs_config.json and update the LVS_SPICE_FILES and LVS_VERILOG_FILES for every design.

Update the other parameters as needed.

In [ ]:
%%shell
cat /content/env
source /content/env
cat > $WORK_ROOT/lvs_config.json <<EOF
{
  "#STD_CELL_LIBRARY": "sky130_fd_sc_hd",
  "#INCLUDE_CONFIGS": [
    "$LVS_ROOT/tech/$PDK/lvs_config.base.json"
  ],
  "TOP_SOURCE": "$PROJECT",
  "TOP_LAYOUT": "$LAYOUT_TOP",
  "EXTRACT_FLATGLOB": [
    "sg13g2_Clamp*",
    "sg13g2_Corner",
    "sg13g2_DCNDiode",
    "sg13g2_DCPDiode",
    "sg13g2_Filler*",
    "sg13g2_GateDecode",
    "sg13g2_IOPad*",
    "sg13g2_LevelDown",
    "sg13g2_LevelUp",
    "sg13g2_RCClamp*",
    "sg13g2_SecondaryProtection",
    "sg13g2_io_*",
    "sg13g2_*LevelUpInv",
    "TEXT*",
    "VIA*"
  ],
  "EXTRACT_ABSTRACT": [ "" ],
  "LVS_FLATTEN": [ "" ],
  "LVS_NOFLATTEN": [ "" ],
  "LVS_IGNORE": [ "" ],
  "LVS_SPICE_FILES": [
    "$SPICE_FILE"
  ],
  "#LVS_VERILOG_FILES": [
    "$VERILOG_FILE"
  ],
  "LAYOUT_FILE": "$LAYOUT_DATA"
}
EOF

Write cvcrc file

In [ ]:
%%shell
cat /content/env
source /content/env
cat > $WORK_ROOT/cvcrc <<EOF
CVC_TOP = $LAYOUT_TOP
CVC_NETLIST = $WORK_ROOT/ext/$LAYOUT_TOP.cdl.gz
CVC_MODEL_FILE = $LVS_ROOT/tech/ihp-sg13g2/cvc.models
CVC_POWER_FILE = $WORK_ROOT/cvc.power.$LAYOUT_TOP
CVC_REPORT_FILE = $WORK_ROOT/cvc.log
EOF

cat > $WORK_ROOT/cvc.power.$PROJECT <<EOF
*vss* power 0.0
iovdd* power 3.3
vdd power 1.5
clk input min@0.0 max@3.3
reset_n input min@0.0 max@3.3
EOF

Create the klayout CDL file from the magic source netlist.

In [ ]:
%%shell
source /content/env
sed -e '/^X[CDLMQR]/s/.//' \
    -e "s/\.subckt $PROJECT/.subckt $LAYOUT_TOP/I" $SPICE_FILE > $CDL_FILE


Results for GPS_LNA


# 11. Greyhound

Not ready - do not run.


Set the project, top cell and gds file names.

In [ ]:
%%shell
source /content/env
cat > $DATA_ROOT/project_env <<'EOF'
export PROJECT=ascon
export LAYOUT_TOP=
export LAYOUT_DATA=design_data/gds/
export WORK_ROOT=$DATA_ROOT/$PROJECT/work/$LAYOUT_TOP
export SPICE_FILE=$WORK_ROOT/$LAYOUT_TOP.spice
export CDL_FILE=$WORK_ROOT/$LAYOUT_TOP.cdl
EOF
cat $DATA_ROOT/project_env
source $DATA_ROOT/project_env
rm -f $DATA_ROOT/work
rm -rf $WORK_ROOT
mkdir -p $WORK_ROOT
ln -s $WORK_ROOT $DATA_ROOT/work
ls -l $DATA_ROOT/work

Create the modified magic source netlist.

In [ ]:
%%shell
cat /content/env
source /content/env
cat > $SPICE_FILE <<EOF
.SUBCKT $LAYOUT_TOP
* no source
.ENDS
EOF

Create lvs_config.json and update the LVS_SPICE_FILES and LVS_VERILOG_FILES for every design.

Update the other parameters as needed.

In [ ]:
%%shell
cat /content/env
source /content/env
cat > $WORK_ROOT/lvs_config.json <<EOF
{
  "#STD_CELL_LIBRARY": "sky130_fd_sc_hd",
  "#INCLUDE_CONFIGS": [
    "$LVS_ROOT/tech/$PDK/lvs_config.base.json"
  ],
  "TOP_SOURCE": "$PROJECT",
  "TOP_LAYOUT": "$LAYOUT_TOP",
  "EXTRACT_FLATGLOB": [
    "sg13g2_Clamp*",
    "sg13g2_Corner",
    "sg13g2_DCNDiode",
    "sg13g2_DCPDiode",
    "sg13g2_Filler*",
    "sg13g2_GateDecode",
    "sg13g2_IOPad*",
    "sg13g2_LevelDown",
    "sg13g2_LevelUp",
    "sg13g2_RCClamp*",
    "sg13g2_SecondaryProtection",
    "sg13g2_io_*",
    "sg13g2_*LevelUpInv",
    "TEXT*",
    "VIA*"
  ],
  "EXTRACT_ABSTRACT": [ "" ],
  "LVS_FLATTEN": [ "" ],
  "LVS_NOFLATTEN": [ "" ],
  "LVS_IGNORE": [ "" ],
  "LVS_SPICE_FILES": [
    "$SPICE_FILE"
  ],
  "#LVS_VERILOG_FILES": [
    "$VERILOG_FILE"
  ],
  "LAYOUT_FILE": "$LAYOUT_DATA"
}
EOF

Write cvcrc file

In [ ]:
%%shell
cat /content/env
source /content/env
cat > $WORK_ROOT/cvcrc <<EOF
CVC_TOP = $LAYOUT_TOP
CVC_NETLIST = $WORK_ROOT/ext/$LAYOUT_TOP.cdl.gz
CVC_MODEL_FILE = $LVS_ROOT/tech/ihp-sg13g2/cvc.models
CVC_POWER_FILE = $WORK_ROOT/cvc.power.$LAYOUT_TOP
CVC_REPORT_FILE = $WORK_ROOT/cvc.log
EOF

cat > $WORK_ROOT/cvc.power.$PROJECT <<EOF
*vss* power 0.0
iovdd* power 3.3
vdd power 1.5
clk input min@0.0 max@3.3
reset_n input min@0.0 max@3.3
EOF

Create the klayout CDL file from the magic source netlist.

In [ ]:
%%shell
source /content/env
sed -e '/^X[CDLMQR]/s/.//' \
    -e "s/\.subckt $PROJECT/.subckt $LAYOUT_TOP/I" $SPICE_FILE > $CDL_FILE


Results for Greyhound


# 12. i2c-gpio-expander

Not ready - do not run.


Set the project, top cell and gds file names.

In [ ]:
%%shell
source /content/env
cat > $DATA_ROOT/project_env <<'EOF'
export PROJECT=ascon
export LAYOUT_TOP=
export LAYOUT_DATA=design_data/gds/
export WORK_ROOT=$DATA_ROOT/$PROJECT/work/$LAYOUT_TOP
export SPICE_FILE=$WORK_ROOT/$LAYOUT_TOP.spice
export CDL_FILE=$WORK_ROOT/$LAYOUT_TOP.cdl
EOF
cat $DATA_ROOT/project_env
source $DATA_ROOT/project_env
rm -f $DATA_ROOT/work
rm -rf $WORK_ROOT
mkdir -p $WORK_ROOT
ln -s $WORK_ROOT $DATA_ROOT/work
ls -l $DATA_ROOT/work

Create the modified magic source netlist.

In [ ]:
%%shell
cat /content/env
source /content/env
cat > $SPICE_FILE <<EOF
.SUBCKT $LAYOUT_TOP
* no source
.ENDS
EOF

Create lvs_config.json and update the LVS_SPICE_FILES and LVS_VERILOG_FILES for every design.

Update the other parameters as needed.

In [ ]:
%%shell
cat /content/env
source /content/env
cat > $WORK_ROOT/lvs_config.json <<EOF
{
  "#STD_CELL_LIBRARY": "sky130_fd_sc_hd",
  "#INCLUDE_CONFIGS": [
    "$LVS_ROOT/tech/$PDK/lvs_config.base.json"
  ],
  "TOP_SOURCE": "$PROJECT",
  "TOP_LAYOUT": "$LAYOUT_TOP",
  "EXTRACT_FLATGLOB": [
    "sg13g2_Clamp*",
    "sg13g2_Corner",
    "sg13g2_DCNDiode",
    "sg13g2_DCPDiode",
    "sg13g2_Filler*",
    "sg13g2_GateDecode",
    "sg13g2_IOPad*",
    "sg13g2_LevelDown",
    "sg13g2_LevelUp",
    "sg13g2_RCClamp*",
    "sg13g2_SecondaryProtection",
    "sg13g2_io_*",
    "sg13g2_*LevelUpInv",
    "TEXT*",
    "VIA*"
  ],
  "EXTRACT_ABSTRACT": [ "" ],
  "LVS_FLATTEN": [ "" ],
  "LVS_NOFLATTEN": [ "" ],
  "LVS_IGNORE": [ "" ],
  "LVS_SPICE_FILES": [
    "$SPICE_FILE"
  ],
  "#LVS_VERILOG_FILES": [
    "$VERILOG_FILE"
  ],
  "LAYOUT_FILE": "$LAYOUT_DATA"
}
EOF

Write cvcrc file

In [ ]:
%%shell
cat /content/env
source /content/env
cat > $WORK_ROOT/cvcrc <<EOF
CVC_TOP = $LAYOUT_TOP
CVC_NETLIST = $WORK_ROOT/ext/$LAYOUT_TOP.cdl.gz
CVC_MODEL_FILE = $LVS_ROOT/tech/ihp-sg13g2/cvc.models
CVC_POWER_FILE = $WORK_ROOT/cvc.power.$LAYOUT_TOP
CVC_REPORT_FILE = $WORK_ROOT/cvc.log
EOF

cat > $WORK_ROOT/cvc.power.$PROJECT <<EOF
*vss* power 0.0
iovdd* power 3.3
vdd power 1.5
clk input min@0.0 max@3.3
reset_n input min@0.0 max@3.3
EOF

Create the klayout CDL file from the magic source netlist.

In [ ]:
%%shell
source /content/env
sed -e '/^X[CDLMQR]/s/.//' \
    -e "s/\.subckt $PROJECT/.subckt $LAYOUT_TOP/I" $SPICE_FILE > $CDL_FILE


Results for i2c-gpio-expander


# 13. Mixer5GHz

Not ready - do not run.


Set the project, top cell and gds file names.

In [ ]:
%%shell
source /content/env
cat > $DATA_ROOT/project_env <<'EOF'
export PROJECT=Mixer5GHz
export LAYOUT_TOP=Mixer5GHz
export LAYOUT_DATA=$DATA_ROOT/$PROJECT/design_data/gds/FMD_QNC_16_Mixer5GHz.mod.gds
# export LAYOUT_DATA=$DATA_ROOT/$PROJECT/design_data/gds/FMD_QNC_16_Mixer5GHz.no_ind.gds
export WORK_ROOT=$DATA_ROOT/$PROJECT/work/$PROJECT
# export SPICE_FILE=$DATA_ROOT/$PROJECT/design_data/xschem/simulations/$PROJECT.spice
# export SPICE_FILE=$CHECK_ROOT/$MPW/$PROJECT/xschem/lvs/$PROJECT.spice
export SPICE_FILE=$WORK_ROOT/$PROJECT.spice
# export CDL_FILE=$DATA_ROOT/$PROJECT/design_data/lvs/$PROJECT.cdl
export CDL_FILE=$WORK_ROOT/$LAYOUT_TOP.cdl
EOF
cat $DATA_ROOT/project_env
source $DATA_ROOT/project_env
rm -f $DATA_ROOT/work
rm -rf $WORK_ROOT
mkdir -p $WORK_ROOT
ln -s $WORK_ROOT $DATA_ROOT/work
ls -l $DATA_ROOT/work

Create the modified magic source netlist.

In [ ]:
%%shell
cat /content/env
source /content/env
# sed -i.bak -e '/^\*\*.\(subckt\|ends\)/s/\*\*//' $SPICE_FILE
cat > $SPICE_FILE <<EOF
** sch_path: /home/shr25031/ihp-mpw-be/TO_Apr2025/Mixer5GHz/xschem/Mixer5GHz.sch
.subckt Mixer5GHz RFN RFP VDC IFP IFN IDC OSCN OSCP VCC ICC GNDC GNDD
*.PININFO RFN:B RFP:B VDC:B IFP:B IFN:B IDC:B OSCN:B OSCP:B VCC:B ICC:B GNDC:B GNDD:B
XM6 RFN LON net2 GNDD sg13_lv_nmos w=60.0u l=0.13u ng=10 m=1
XRL2 RFN VDC rppd w=4.50e-6 l=3.20e-6 m=1 b=0
XRL1 RFP VDC rppd w=4.50e-6 l=3.20e-6 m=1 b=0
XM8 RFN LOP net3 GNDD sg13_lv_nmos w=60.0u l=0.13u ng=10 m=1
XM7 RFP LON net3 GNDD sg13_lv_nmos w=60.0u l=0.13u ng=10 m=1
XM5 RFP LOP net2 GNDD sg13_lv_nmos w=60.0u l=0.13u ng=10 m=1
XM4 net3 IFN net1 GNDD sg13_lv_nmos w=90.0u l=0.13u ng=15 m=1
XM3 net2 IFP net1 GNDD sg13_lv_nmos w=90.0u l=0.13u ng=15 m=1
XM2 IDC IDC GNDD GNDD sg13_lv_nmos w=120.0u l=0.13u ng=20 m=1
XM1 net1 IDC GNDD GNDD sg13_lv_nmos w=120.0u l=0.13u ng=20 m=1
XM11 LOP LON net4 GNDD sg13_lv_nmos w=90.0u l=0.13u ng=15 m=1
XM12 LON LOP net4 GNDD sg13_lv_nmos w=90.0u l=0.13u ng=15 m=1
XM9 IDC IDC GNDD GNDD sg13_lv_nmos w=120.0u l=0.13u ng=20 m=1
XM10 net4 IDC GNDD GNDD sg13_lv_nmos w=120.0u l=0.13u ng=20 m=1
XC1 VDC LOP cap_cmim w=11.745e-6 l=9.445e-6 m=1
XR1 LOP VDC rppd w=4.4e-6 l=1.5e-6 m=1 b=0
XC2 VDC LON cap_cmim w=11.745e-6 l=9.445e-6 m=1
XR3 LON VDC rppd w=4.4e-6 l=1.5e-6 m=1 b=0
XM13 OSCP OSCN net5 GNDC sg13_lv_nmos w=90.0u l=0.13u ng=15 m=1
XM14 OSCN OSCP net5 GNDC sg13_lv_nmos w=90.0u l=0.13u ng=15 m=1
XM15 ICC ICC GNDC GNDC sg13_lv_nmos w=120.0u l=0.13u ng=20 m=1
XM16 net5 ICC GNDC GNDC sg13_lv_nmos w=120.0u l=0.13u ng=20 m=1
XC3 VCC OSCP cap_cmim w=19.1e-6 l=10.7e-6 m=1
XR3 OSCP VCC rppd w=4.35e-6 l=1.5e-6 m=1 b=0
XC4 VCC OSCN cap_cmim w=19.1e-6 l=10.7e-6 m=1
XR4 OSCN VCC rppd w=4.35e-6 l=1.5e-6 m=1 b=0
XL1 VDC LOP GNDD inductor2 w=10u s=1 d=222u a=1.596788e-08 nr=2 m=1
XL2 VDC LON GNDD inductor2 w=10u s=1 d=222u a=1.596788e-08 nr=2 m=1
XL3 VCC OSCP GNDC inductor2 w=10u s=1 d=222u a=1.596788e-08 nr=2 m=1
XL4 VCC OSCN GNDC inductor2 w=10u s=1 d=222u a=1.596788e-08 nr=2 m=1
.ends
EOF

Create lvs_config.json and update the LVS_SPICE_FILES and LVS_VERILOG_FILES for every design.

Update the other parameters as needed.

In [ ]:
%%shell
cat /content/env
source /content/env
cat > $WORK_ROOT/lvs_config.json <<EOF
{
  "#STD_CELL_LIBRARY": "sky130_fd_sc_hd",
  "#INCLUDE_CONFIGS": [
    "$LVS_ROOT/tech/$PDK/lvs_config.base.json"
  ],
  "TOP_SOURCE": "$PROJECT",
  "TOP_LAYOUT": "$LAYOUT_TOP",
  "EXTRACT_FLATGLOB": [
    "*_FILL_*",
    "PADs",
    "cmim*",
    "nmos*",
    "rppd*",
    "inductor*",
    "sealring*"
  ],
  "EXTRACT_ABSTRACT": [ "" ],
  "LVS_FLATTEN": [ "" ],
  "LVS_NOFLATTEN": [ "" ],
  "LVS_IGNORE": [ "" ],
  "LVS_SPICE_FILES": [
    "$SPICE_FILE"
  ],
  "#LVS_VERILOG_FILES": [
    "$VERILOG_FILE"
  ],
  "LAYOUT_FILE": "$LAYOUT_DATA"
}
EOF

Write cvcrc file

In [ ]:
%%shell
cat /content/env
source /content/env
cat > $WORK_ROOT/cvcrc <<EOF
CVC_TOP = $LAYOUT_TOP
CVC_NETLIST = $WORK_ROOT/ext/$LAYOUT_TOP.cdl.gz
CVC_MODEL_FILE = $LVS_ROOT/tech/ihp-sg13g2/cvc.models
CVC_POWER_FILE = $WORK_ROOT/cvc.power.$LAYOUT_TOP
CVC_REPORT_FILE = $WORK_ROOT/cvc.log
EOF

cat > $WORK_ROOT/cvc.power.$PROJECT <<EOF
GNDC power 0.0
GNDD power 0.0
VCC power 1.65
VDC power 1.65
ICC input min@0.0 max@1.65
IDC input min@0.0 max@1.65
IFP input min@0.0 max@1.65
IFN input min@0.0 max@1.65
EOF

Create the klayout CDL file from the magic source netlist.

In [ ]:
%%shell
source /content/env
sed -e '/^X[CDLMQR]/s/.//' \
    -e "s/\.subckt $PROJECT/.subckt $LAYOUT_TOP/I" $SPICE_FILE > $CDL_FILE


Results for Mixer5GHz


# 14. PA_180GHz

Checks will not work with GDS file in repo.
1. Add text ports `RFIN` `RFOUT` `VCC1A` `VCC2A` `VCC1B` `VCC2B` `VBB1` `VBB2` `VSS` on layer 134/25 to top layout.
1. Copy corresponding pins shapes on layer 134/2 from `PA` to top layout (magic only).
1. Copy Passiv shapes layer 9/0 over `RFIN` and `RFOUT` pins to the top layout and move to dfpad - 41/0 (magic only).
1. Remove cells `NoFillerStack$1` `NoFillerStack$2` `NoFillerStack$3` placed in the top level.  These cells cause defect in extracting some devices (magic only).

Requires updated setup file `ihp-sg13g2_setup.tcl` to follow netgen `Commit 0bee21c`

Set the project, top cell and gds file names.

In [ ]:
%%shell
source /content/env
cat > $DATA_ROOT/project_env <<'EOF'
export PROJECT=PA_180GHz
export LAYOUT_TOP=FMD_QNC_07a_2way_PA
export LAYOUT_DATA=$DATA_ROOT/$PROJECT/design_data/gds/FMD_QNC_07a_20dBm_Psat_two_way_power_amp_180GHz.mod.gds
export WORK_ROOT=$DATA_ROOT/$PROJECT/work/$PROJECT
# export SPICE_FILE=$DATA_ROOT/$PROJECT/design_data/xschem/simulations/$PROJECT.spice
# export SPICE_FILE=$CHECK_ROOT/$MPW/$PROJECT/xschem/lvs/$PROJECT.spice
export SPICE_FILE=$WORK_ROOT/$PROJECT.spice
# export CDL_FILE=$DATA_ROOT/$PROJECT/design_data/lvs/$PROJECT.cdl
export CDL_FILE=$WORK_ROOT/$LAYOUT_TOP.cdl
EOF
# cat $DATA_ROOT/project_env
source $DATA_ROOT/project_env
if [[ -f $LAYOUT_DATA ]]; then
  ls -l $LAYOUT_DATA
else
  echo "LAYOUT_DATA=$LAYOUT_DATA does not exist"
  exit 1
fi
rm -f $DATA_ROOT/work
rm -rf $WORK_ROOT
mkdir -p $WORK_ROOT
ln -s $WORK_ROOT $DATA_ROOT/work
ls -l $DATA_ROOT/work

Create the modified magic source netlist.

In [ ]:
%%shell
cat /content/env
source /content/env
cat > $SPICE_FILE <<EOF
** sch_path: /home/shr25031/ihp-mpw-be/TO_Apr2025/PA_180GHz/xschem/PA_180GHz.sch
.subckt PA_180GHz VBB1 VBB2 RFIN VCC1B VCC2B VSS RFOUT VCC1A VCC2A
*.PININFO VBB1:B VBB2:B RFIN:I VCC1B:B VCC2B:B VSS:B RFOUT:O VCC1A:B VCC2A:B
XQnpn13G1 _net1 _net0 netq1 VSS npn13G2 le=900e-9 we=70.0n m=8
XQnpn13G2 _net1 _net0 netq1 VSS npn13G2 le=900e-9 we=70.0n m=8
XRrsil1 VBB1 _net0 rsil w=2e-6 l=11e-6 m=1 b=0
XCcap_rfcmim1 RFIN _net0 VSS rfcmim w=4.6e-6 l=4.2e-6 wfeed=3.0e-6
XRrppd2 _net1 VCC1B rppd w=35e-6 l=0.5e-6 m=1 b=0
XCcap_rfcmim3 _net1 _net2 VSS rfcmim w=6.6e-6 l=6.1e-6 wfeed=3.0e-6
XQnpn13G3 _net3 _net7 netq3 VSS npn13G2 le=900e-9 we=70.0n m=8
XQnpn13G4 _net3 _net7 netq3 VSS npn13G2 le=900e-9 we=70.0n m=8
XCcap_rfcmim4 _net2 _net7 VSS rfcmim w=4.6e-6 l=4.5e-6 wfeed=3.0e-6
XRrsil2 VBB1 _net7 rsil w=2e-6 l=11e-6 m=1 b=0
XRrppd1 _net3 VCC1B rppd w=35e-6 l=0.5e-6 m=1 b=0
XQnpn13G5 _net6 _net5 netq6 VSS npn13G2 le=900e-9 we=70.0n m=10
XQnpn13G6 _net6 _net5 netq6 VSS npn13G2 le=900e-9 we=70.0n m=10
XCcap_rfcmim5 _net3 _net4 VSS rfcmim w=6.6e-6 l=5.3e-6 wfeed=3.0e-6
XCcap_rfcmim6 _net4 _net5 VSS rfcmim w=4.9e-6 l=4.3e-6 wfeed=3.0e-6
XRrppd3 _net6 VCC2B rppd w=35e-6 l=0.5e-6 m=1 b=0
XRrsil3 VBB2 _net5 rsil w=2e-6 l=12e-6 m=1 b=0
XCcap_rfcmim7 _net6 RFOUT VSS rfcmim w=5.2e-6 l=4.8e-6 wfeed=3.0e-6
XQnpn13G7 _net9 _net8 netq9 VSS npn13G2 le=900e-9 we=70.0n m=8
XQnpn13G8 _net9 _net8 netq9 VSS npn13G2 le=900e-9 we=70.0n m=8
XRrsil7 VBB1 _net8 rsil w=2e-6 l=11e-6 m=1 b=0
XCcap_rfcmim8 RFIN _net8 VSS rfcmim w=4.6e-6 l=4.2e-6 wfeed=3.0e-6
XRrppd4 _net9 VCC1A rppd w=35e-6 l=0.5e-6 m=1 b=0
XCcap_rfcmim9 _net9 _net10 VSS rfcmim w=6.6e-6 l=6.1e-6 wfeed=3.0e-6
XQnpn13G9 _net11 _net15 netq11 VSS npn13G2 le=900e-9 we=70.0n m=8
XQnpn13G10 _net11 _net15 netq11 VSS npn13G2 le=900e-9 we=70.0n m=8
XCcap_rfcmim10 _net10 _net15 VSS rfcmim w=4.6e-6 l=4.5e-6 wfeed=3.0e-6
XRrsil8 VBB1 _net15 rsil w=2e-6 l=11e-6 m=1 b=0
XRrppd5 _net11 VCC1A rppd w=35e-6 l=0.5e-6 m=1 b=0
XQnpn13G11 _net14 _net13 netq14 VSS npn13G2 le=900e-9 we=70.0n m=10
XQnpn13G12 _net14 _net13 netq14 VSS npn13G2 le=900e-9 we=70.0n m=10
XCcap_rfcmim11 _net11 _net12 VSS rfcmim w=6.6e-6 l=5.3e-6 wfeed=3.0e-6
XCcap_rfcmim12 _net12 _net13 VSS rfcmim w=4.9e-6 l=4.3e-6 wfeed=3.0e-6
XRrppd6 _net14 VCC2A rppd w=35e-6 l=0.5e-6 m=1 b=0
XRrsil9 VBB2 _net13 rsil w=2e-6 l=12e-6 m=1 b=0
XCcap_rfcmim13 _net14 RFOUT VSS rfcmim w=5.2e-6 l=4.8e-6 wfeed=3.0e-6
XCcap_cmim1 VCC1A VSS cap_cmim w=10.5e-6 l=15.0e-6 m=1
XCcap_cmim2 VCC1A VSS cap_cmim w=10.5e-6 l=14.885e-6 m=1
XCcap_cmim3 VCC1A VSS cap_cmim w=10.5e-6 l=15.0e-6 m=1
XCcap_cmim4 VCC1A VSS cap_cmim w=10.5e-6 l=15.0e-6 m=1
XCcap_cmim5 VCC2A VSS cap_cmim w=10.5e-6 l=15.0e-6 m=1
XCcap_cmim6 VCC2A VSS cap_cmim w=10.5e-6 l=15.0e-6 m=1
XCcap_cmim7 VCC1B VSS cap_cmim w=10.5e-6 l=15.0e-6 m=1
XCcap_cmim8 VCC1B VSS cap_cmim w=10.5e-6 l=14.885e-6 m=1
XCcap_cmim9 VCC1B VSS cap_cmim w=10.5e-6 l=15.0e-6 m=1
XCcap_cmim10 VCC1B VSS cap_cmim w=10.5e-6 l=15.0e-6 m=1
XCcap_cmim11 VCC2B VSS cap_cmim w=10.5e-6 l=15.0e-6 m=1
XCcap_cmim12 VCC2B VSS cap_cmim w=10.5e-6 l=15.0e-6 m=1
XRrsil14 RFOUT RFOUT rsil w=2.04e-6 l=28e-6 m=1 b=0
XRrsil13 RFIN RFIN rsil w=2.04e-6 l=28e-6 m=1 b=0
RRFIN VSS RFIN res_topmetal2 w=15e-6 l=40.6e-6 m=2
RRFOUT VSS RFOUT res_topmetal2 w=15e-6 l=73.1e-6 m=2
RRM1 VSS netq1 res_topmetal2 w=15e-6 l=7.1e-6 m=1
RRM2 VSS netq3 res_topmetal2 w=15e-6 l=7.1e-6 m=1
RRM3 VSS netq6 res_topmetal2 w=15e-6 l=7.1e-6 m=1
RRM4 VSS _net2 res_topmetal2 w=15e-6 l=18.1e-6 m=1
RRM5 VSS _net4 res_topmetal2 w=15e-6 l=18.1e-6 m=1
RRM6 VSS netq9 res_topmetal2 w=15e-6 l=7.1e-6 m=1
RRM7 VSS netq11 res_topmetal2 w=15e-6 l=7.1e-6 m=1
RRM8 VSS netq14 res_topmetal2 w=15e-6 l=7.1e-6 m=1
RRM9 VSS _net10 res_topmetal2 w=15e-6 l=18.1e-6 m=1
RRM10 VSS _net12 res_topmetal2 w=15e-6 l=18.1e-6 m=1
.ends
EOF

Create lvs_config.json and update the LVS_SPICE_FILES and LVS_VERILOG_FILES for every design.

Update the other parameters as needed.

In [ ]:
%%shell
cat /content/env
source /content/env
cat > $WORK_ROOT/lvs_config.json <<EOF
{
  "#STD_CELL_LIBRARY": "sky130_fd_sc_hd",
  "#INCLUDE_CONFIGS": [
    "$LVS_ROOT/tech/$PDK/lvs_config.base.json"
  ],
  "TOP_SOURCE": "$PROJECT",
  "TOP_LAYOUT": "$LAYOUT_TOP",
  "EXTRACT_FLATGLOB": [
    "cmim*",
    "npn13G2*",
    "rfcmim*",
    "rppd*",
    "rsil*",
    "via_stack*",
    "sealring*",
    "NoFillerStack*",
    "*_FILL_*",
    "GND",
    "PA"
  ],
  "EXTRACT_ABSTRACT": [ "" ],
  "LVS_FLATTEN": [ "" ],
  "LVS_NOFLATTEN": [ "" ],
  "LVS_IGNORE": [ "" ],
  "LVS_SPICE_FILES": [
    "$SPICE_FILE"
  ],
  "#LVS_VERILOG_FILES": [
    "$VERILOG_FILE"
  ],
  "LAYOUT_FILE": "$LAYOUT_DATA"
}
EOF

Write cvcrc file

In [ ]:
%%shell
cat /content/env
source /content/env
cat > $WORK_ROOT/cvcrc <<EOF
CVC_TOP = $LAYOUT_TOP
CVC_NETLIST = $WORK_ROOT/ext/$LAYOUT_TOP.cdl.gz
CVC_MODEL_FILE = $LVS_ROOT/tech/ihp-sg13g2/cvc.models
CVC_POWER_FILE = $WORK_ROOT/cvc.power.$LAYOUT_TOP
CVC_REPORT_FILE = $WORK_ROOT/cvc.log
EOF

cat > $WORK_ROOT/cvc.power.$PROJECT <<EOF
VSS power 0.0
VBB1 power 0.97
VBB2 power 0.94
VCC1A power 1.7
VCC1B power 1.7
VCC2A power 1.8
VCC2B power 1.8
RFIN min@0.94 max@1.8
EOF

Create the klayout CDL file from the magic source netlist.

In [ ]:
%%shell
source /content/env
sed -e '/^X[CDLMQR]/s/.//' \
    -e "s/\.subckt $PROJECT/.subckt $LAYOUT_TOP/I" $SPICE_FILE > $CDL_FILE


Results for PA_180GHz

1. Ignored CVC errors at metal resistor:
```
! Short Detected:
/R7 R res_topmetal2 w=15u l=40.6u (r=1)
S: RFIN    Sim: RFIN
D: VSS     Sim: VSS@0 r=0

! Short Detected:
/R3 R res_topmetal2 w=15u l=40.6u (r=1)
S: VSS    Sim: VSS@0 r=0
D: RFIN    Sim: RFIN
```




# 15. TTIHP0p2

Not ready - do not run.


Set the project, top cell and gds file names.

In [ ]:
%%shell
source /content/env
cat > $DATA_ROOT/project_env <<'EOF'
export PROJECT=ascon
export LAYOUT_TOP=
export LAYOUT_DATA=design_data/gds/
export WORK_ROOT=$DATA_ROOT/$PROJECT/work/$LAYOUT_TOP
export SPICE_FILE=$WORK_ROOT/$LAYOUT_TOP.spice
export CDL_FILE=$WORK_ROOT/$LAYOUT_TOP.cdl
EOF
cat $DATA_ROOT/project_env
source $DATA_ROOT/project_env
rm -f $DATA_ROOT/work
rm -rf $WORK_ROOT
mkdir -p $WORK_ROOT
ln -s $WORK_ROOT $DATA_ROOT/work
ls -l $DATA_ROOT/work

Create the modified magic source netlist.

In [ ]:
%%shell
cat /content/env
source /content/env
cat > $SPICE_FILE <<EOF
.SUBCKT $LAYOUT_TOP
* no source
.ENDS
EOF

Create lvs_config.json and update the LVS_SPICE_FILES and LVS_VERILOG_FILES for every design.

Update the other parameters as needed.

In [ ]:
%%shell
cat /content/env
source /content/env
cat > $WORK_ROOT/lvs_config.json <<EOF
{
  "#STD_CELL_LIBRARY": "sky130_fd_sc_hd",
  "#INCLUDE_CONFIGS": [
    "$LVS_ROOT/tech/$PDK/lvs_config.base.json"
  ],
  "TOP_SOURCE": "$PROJECT",
  "TOP_LAYOUT": "$LAYOUT_TOP",
  "EXTRACT_FLATGLOB": [
    "sg13g2_Clamp*",
    "sg13g2_Corner",
    "sg13g2_DCNDiode",
    "sg13g2_DCPDiode",
    "sg13g2_Filler*",
    "sg13g2_GateDecode",
    "sg13g2_IOPad*",
    "sg13g2_LevelDown",
    "sg13g2_LevelUp",
    "sg13g2_RCClamp*",
    "sg13g2_SecondaryProtection",
    "sg13g2_io_*",
    "sg13g2_*LevelUpInv",
    "TEXT*",
    "VIA*"
  ],
  "EXTRACT_ABSTRACT": [ "" ],
  "LVS_FLATTEN": [ "" ],
  "LVS_NOFLATTEN": [ "" ],
  "LVS_IGNORE": [ "" ],
  "LVS_SPICE_FILES": [
    "$SPICE_FILE"
  ],
  "#LVS_VERILOG_FILES": [
    "$VERILOG_FILE"
  ],
  "LAYOUT_FILE": "$LAYOUT_DATA"
}
EOF

Write cvcrc file

In [ ]:
%%shell
cat /content/env
source /content/env
cat > $WORK_ROOT/cvcrc <<EOF
CVC_TOP = $LAYOUT_TOP
CVC_NETLIST = $WORK_ROOT/ext/$LAYOUT_TOP.cdl.gz
CVC_MODEL_FILE = $LVS_ROOT/tech/ihp-sg13g2/cvc.models
CVC_POWER_FILE = $WORK_ROOT/cvc.power.$LAYOUT_TOP
CVC_REPORT_FILE = $WORK_ROOT/cvc.log
EOF

cat > $WORK_ROOT/cvc.power.$PROJECT <<EOF
*vss* power 0.0
iovdd* power 3.3
vdd power 1.5
clk input min@0.0 max@3.3
reset_n input min@0.0 max@3.3
EOF

Create the klayout CDL file from the magic source netlist.

In [ ]:
%%shell
source /content/env
sed -e '/^X[CDLMQR]/s/.//' \
    -e "s/\.subckt $PROJECT/.subckt $LAYOUT_TOP/I" $SPICE_FILE > $CDL_FILE


Results for TTIHP0p2


# 16. TTIHP25a

Not ready - do not run.


Set the project, top cell and gds file names.

In [ ]:
%%shell
source /content/env
cat > $DATA_ROOT/project_env <<'EOF'
export PROJECT=ascon
export LAYOUT_TOP=
export LAYOUT_DATA=design_data/gds/
export WORK_ROOT=$DATA_ROOT/$PROJECT/work/$LAYOUT_TOP
export SPICE_FILE=$WORK_ROOT/$LAYOUT_TOP.spice
export CDL_FILE=$WORK_ROOT/$LAYOUT_TOP.cdl
EOF
cat $DATA_ROOT/project_env
source $DATA_ROOT/project_env
rm -f $DATA_ROOT/work
rm -rf $WORK_ROOT
mkdir -p $WORK_ROOT
ln -s $WORK_ROOT $DATA_ROOT/work
ls -l $DATA_ROOT/work

Create the modified magic source netlist.

In [ ]:
%%shell
cat /content/env
source /content/env
cat > $SPICE_FILE <<EOF
.SUBCKT $LAYOUT_TOP
* no source
.ENDS
EOF

Create lvs_config.json and update the LVS_SPICE_FILES and LVS_VERILOG_FILES for every design.

Update the other parameters as needed.

In [ ]:
%%shell
cat /content/env
source /content/env
cat > $WORK_ROOT/lvs_config.json <<EOF
{
  "#STD_CELL_LIBRARY": "sky130_fd_sc_hd",
  "#INCLUDE_CONFIGS": [
    "$LVS_ROOT/tech/$PDK/lvs_config.base.json"
  ],
  "TOP_SOURCE": "$PROJECT",
  "TOP_LAYOUT": "$LAYOUT_TOP",
  "EXTRACT_FLATGLOB": [
    "sg13g2_Clamp*",
    "sg13g2_Corner",
    "sg13g2_DCNDiode",
    "sg13g2_DCPDiode",
    "sg13g2_Filler*",
    "sg13g2_GateDecode",
    "sg13g2_IOPad*",
    "sg13g2_LevelDown",
    "sg13g2_LevelUp",
    "sg13g2_RCClamp*",
    "sg13g2_SecondaryProtection",
    "sg13g2_io_*",
    "sg13g2_*LevelUpInv",
    "TEXT*",
    "VIA*"
  ],
  "EXTRACT_ABSTRACT": [ "" ],
  "LVS_FLATTEN": [ "" ],
  "LVS_NOFLATTEN": [ "" ],
  "LVS_IGNORE": [ "" ],
  "LVS_SPICE_FILES": [
    "$SPICE_FILE"
  ],
  "#LVS_VERILOG_FILES": [
    "$VERILOG_FILE"
  ],
  "LAYOUT_FILE": "$LAYOUT_DATA"
}
EOF

Write cvcrc file

In [ ]:
%%shell
cat /content/env
source /content/env
cat > $WORK_ROOT/cvcrc <<EOF
CVC_TOP = $LAYOUT_TOP
CVC_NETLIST = $WORK_ROOT/ext/$LAYOUT_TOP.cdl.gz
CVC_MODEL_FILE = $LVS_ROOT/tech/ihp-sg13g2/cvc.models
CVC_POWER_FILE = $WORK_ROOT/cvc.power.$LAYOUT_TOP
CVC_REPORT_FILE = $WORK_ROOT/cvc.log
EOF

cat > $WORK_ROOT/cvc.power.$PROJECT <<EOF
*vss* power 0.0
iovdd* power 3.3
vdd power 1.5
clk input min@0.0 max@3.3
reset_n input min@0.0 max@3.3
EOF

Create the klayout CDL file from the magic source netlist.

In [ ]:
%%shell
source /content/env
sed -e '/^X[CDLMQR]/s/.//' \
    -e "s/\.subckt $PROJECT/.subckt $LAYOUT_TOP/I" $SPICE_FILE > $CDL_FILE


Results for TTIHP25a


# 17. VCO_130nm_LSI

Not ready - do not run.


Set the project, top cell and gds file names.

In [ ]:
%%shell
source /content/env
cat > $DATA_ROOT/project_env <<'EOF'
export PROJECT=ascon
export LAYOUT_TOP=
export LAYOUT_DATA=design_data/gds/
export WORK_ROOT=$DATA_ROOT/$PROJECT/work/$LAYOUT_TOP
export SPICE_FILE=$WORK_ROOT/$LAYOUT_TOP.spice
export CDL_FILE=$WORK_ROOT/$LAYOUT_TOP.cdl
EOF
cat $DATA_ROOT/project_env
source $DATA_ROOT/project_env
rm -f $DATA_ROOT/work
rm -rf $WORK_ROOT
mkdir -p $WORK_ROOT
ln -s $WORK_ROOT $DATA_ROOT/work
ls -l $DATA_ROOT/work

Create the modified magic source netlist.

In [ ]:
%%shell
cat /content/env
source /content/env
cat > $SPICE_FILE <<EOF
.SUBCKT $LAYOUT_TOP
* no source
.ENDS
EOF

Create lvs_config.json and update the LVS_SPICE_FILES and LVS_VERILOG_FILES for every design.

Update the other parameters as needed.

In [ ]:
%%shell
cat /content/env
source /content/env
cat > $WORK_ROOT/lvs_config.json <<EOF
{
  "#STD_CELL_LIBRARY": "sky130_fd_sc_hd",
  "#INCLUDE_CONFIGS": [
    "$LVS_ROOT/tech/$PDK/lvs_config.base.json"
  ],
  "TOP_SOURCE": "$PROJECT",
  "TOP_LAYOUT": "$LAYOUT_TOP",
  "EXTRACT_FLATGLOB": [
    "sg13g2_Clamp*",
    "sg13g2_Corner",
    "sg13g2_DCNDiode",
    "sg13g2_DCPDiode",
    "sg13g2_Filler*",
    "sg13g2_GateDecode",
    "sg13g2_IOPad*",
    "sg13g2_LevelDown",
    "sg13g2_LevelUp",
    "sg13g2_RCClamp*",
    "sg13g2_SecondaryProtection",
    "sg13g2_io_*",
    "sg13g2_*LevelUpInv",
    "TEXT*",
    "VIA*"
  ],
  "EXTRACT_ABSTRACT": [ "" ],
  "LVS_FLATTEN": [ "" ],
  "LVS_NOFLATTEN": [ "" ],
  "LVS_IGNORE": [ "" ],
  "LVS_SPICE_FILES": [
    "$SPICE_FILE"
  ],
  "#LVS_VERILOG_FILES": [
    "$VERILOG_FILE"
  ],
  "LAYOUT_FILE": "$LAYOUT_DATA"
}
EOF

Write cvcrc file

In [ ]:
%%shell
cat /content/env
source /content/env
cat > $WORK_ROOT/cvcrc <<EOF
CVC_TOP = $LAYOUT_TOP
CVC_NETLIST = $WORK_ROOT/ext/$LAYOUT_TOP.cdl.gz
CVC_MODEL_FILE = $LVS_ROOT/tech/ihp-sg13g2/cvc.models
CVC_POWER_FILE = $WORK_ROOT/cvc.power.$LAYOUT_TOP
CVC_REPORT_FILE = $WORK_ROOT/cvc.log
EOF

cat > $WORK_ROOT/cvc.power.$PROJECT <<EOF
*vss* power 0.0
iovdd* power 3.3
vdd power 1.5
clk input min@0.0 max@3.3
reset_n input min@0.0 max@3.3
EOF

Create the klayout CDL file from the magic source netlist.

In [ ]:
%%shell
source /content/env
sed -e '/^X[CDLMQR]/s/.//' \
    -e "s/\.subckt $PROJECT/.subckt $LAYOUT_TOP/I" $SPICE_FILE > $CDL_FILE


Results for VCO_130nm_LSI


+++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Run BE checks


In [ ]:
%%shell
cat /content/env
source /content/env
# run_be_checks includes magic/netgen soft connection and LVS, and CVC-RV
$LVS_ROOT/run_be_checks $WORK_ROOT/lvs_config.json

python3 $PDK_ROOT/$PDK/libs.tech/klayout/tech/lvs/run_lvs.py --layout=$LAYOUT_DATA \
  --netlist=$CDL_FILE \
  --run_dir=$WORK_ROOT/klayout \
  --topcell=$LAYOUT_TOP \
  --run_mode=deep \
  --spice_comments \
  --no_simplify \
  --combine_devices \
  --top_lvl_pins

LVS_BASENAME=$( echo $LAYOUT_DATA | sed -e 's,.*/,,' -e 's,\..*,,' )
$LVS_ROOT/check_klayout_ports $WORK_ROOT/klayout/$LVS_BASENAME.lvsdb > $WORK_ROOT/klayout/port_check.log
if [[ -e $WORK_ROOT/klayout/port_check.log ]]; then
  if [[ -s $WORK_ROOT/klayout/port_check.log ]]; then
    echo "* Klayout ports do not match *"
    echo "layout     source"
    awk '{printf "%-10s %s\n", $1, $2}' $WORK_ROOT/klayout/port_check.log
  else
    echo "Klayout ports match"
  fi
fi

# View verification logs

In [ ]:
from google.colab import files

files.view('/content/data/work/soft.log')
files.view('/content/data/work/lvs.log')
files.view('/content/data/work/cvc.log')

# View soft verification report

In [ ]:
from google.colab import files

files.view('/content/data/work/soft.report')

# View LVS report

In [ ]:
from google.colab import files

files.view('/content/data/work/lvs.report')

# View CVC report

In [ ]:
from google.colab import files

files.view('/content/data/work/cvc.error')